# Rates Factor Timing Engine — v0.8
## Empirical Bond-Premia Predictors: Fama–Bliss + Cochrane–Piazzesi Style + Multi-Horizon Timing

v0.7 established a stable funded-factor / portfolio-construction stack and showed that the generic macro/curve state model did **not** materially outperform an unconditional premium estimate.

v0.8 therefore changes the **asset-pricing signal model**, not the factor construction or portfolio machinery.

The new research layer introduces:

1. **Fama–Bliss forward spreads**

\[
FB_t^{(n)} = f_t^{(n)} - y_t^{(1)},
\]

with annual forward rates implied by the zero curve:

\[
f_t^{(n)} = n y_t^{(n)} - (n-1)y_t^{(n-1)}.
\]

2. **Cochrane–Piazzesi-style common return predictor**

A walk-forward linear combination of the 1Y–5Y forward curve is estimated to forecast the average future Treasury excess return. The fitted value is then used as a scalar predictor of the Level/Slope/Curvature factor premia.

3. **Forecast horizons**

\[
H \in \{3,6,12\}\text{ months}.
\]

Signals are forecast at these horizons but the portfolio is still rebalanced monthly. Therefore the horizon controls the **expected-return signal**, not the trading frequency.

4. **Prespecified model families**

- `BASE_STATE`: v0.7 macro/curve state benchmark;
- `FB`: Fama–Bliss forward spreads;
- `FB_STATE`: forward spreads + v0.7 state;
- `CP`: Cochrane–Piazzesi-style common predictor;
- `CP_STATE`: CP predictor + v0.7 state.

The stable downstream stack remains unchanged:

\[
\text{funded L/S/C portfolios}
\rightarrow
\text{carry + expected curve repricing}
\rightarrow
\text{mean-confidence gate}
\rightarrow
\text{risk-scaled factor allocation}
\rightarrow
\text{maturity overlay}
\rightarrow
\text{realized P\&L}.
\]

No model family or horizon is selected by tuning on realized strategy P&L in this notebook. They are compared as a prespecified empirical-asset-pricing research grid.


## 0. Environment

In [ ]:
import sys
import subprocess
import importlib.util
from pathlib import Path

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "pandas_datareader": "pandas_datareader",
    "statsmodels": "statsmodels",
    "yfinance": "yfinance",
}

missing = [
    pip_name
    for module_name, pip_name in required.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

from scipy.stats import norm
from pandas_datareader import data as pdr
from dataclasses import dataclass
from math import sqrt
from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
np.set_printoptions(suppress=True, precision=6)

print("Environment ready.")


## 1. Configuration

In [ ]:
# ------------------------------------------------------------------
# Data
# ------------------------------------------------------------------
DATA_START = "1990-01-01"
DATA_END = None

CACHE_DIR = Path("./rates_factor_timing_cache")
CACHE_DIR.mkdir(exist_ok=True)

REFRESH_DATA = False

ZERO_SERIES = {
    1: "THREEFY1",
    2: "THREEFY2",
    3: "THREEFY3",
    4: "THREEFY4",
    5: "THREEFY5",
    6: "THREEFY6",
    7: "THREEFY7",
    8: "THREEFY8",
    9: "THREEFY9",
    10: "THREEFY10",
}

TRADED_MATURITIES = list(range(2, 11))

STATE_SERIES = {
    "CPI": "CPIAUCSL",
    "INDPRO": "INDPRO",
    "FEDFUNDS": "FEDFUNDS",
    "BREAKEVEN10Y": "T10YIE",
}

# ------------------------------------------------------------------
# Curve factor extraction
# ------------------------------------------------------------------
EWMA_LAMBDA = 0.97
EWMA_WARMUP_MONTHS = 36

PCA_TRAIN_END = "2002-12-31"
N_FACTORS = 3
FACTOR_NAMES = ["Level", "Slope", "Curvature"]

# ------------------------------------------------------------------
# Factor-mimicking portfolios
# ------------------------------------------------------------------
# Structural regularization only; do NOT tune this on strategy P&L.
MIMIC_NUMERICAL_RIDGE = 1e-8

# Funded factor portfolios: exact cross-factor neutrality, explicit cash funding.
MIMIC_GROSS = 1.0

# Objective covariance: trailing maturity-return covariance with diagonal shrinkage.
MIMIC_COV_LOOKBACK = 60
MIMIC_COV_MIN_OBS = 36
MIMIC_COV_DIAG_SHRINK = 0.25

# ------------------------------------------------------------------
# Conditional forecasting
# ------------------------------------------------------------------
COND_MIN_TRAIN = 60
COND_WINDOW = None
COND_RIDGE = 1.0
FORECAST_HORIZON = 1

# Confidence that the conditional MEAN, not the next realized return,
# has the indicated sign.
MEAN_CONFIDENCE_THRESHOLD = 0.60

# No arbitrary volatility-scaled hurdle in the baseline.
# Transaction costs are assessed explicitly in the backtest.
EXPECTED_RETURN_HURDLE = 0.0

# ------------------------------------------------------------------
# Overlay portfolio
# ------------------------------------------------------------------
FACTOR_VOL_LOOKBACK = 36
FACTOR_VOL_MIN_OBS = 24

MAX_FACTOR_ALLOCATION = 0.75
TOTAL_FACTOR_GROSS = 1.00
MAX_UNDERLYING_GROSS = 1.25

CORE_OVERLAY_SCALE = 0.50

BASE_COST_BPS = 5.0
COST_SENSITIVITY_BPS = [1.0, 5.0, 10.0]

# ------------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------------
RUN_FMB_DIAGNOSTIC = True



# ------------------------------------------------------------------
# v0.8 empirical bond-premia research grid
# ------------------------------------------------------------------
PREDICTION_HORIZONS = [3, 6, 12]

MODEL_FAMILIES = [
    "BASE_STATE",
    "FB",
    "FB_STATE",
    "CP",
    "CP_STATE",
]

# Original Cochrane–Piazzesi-style forward-curve block: 1Y through 5Y.
CP_FORWARD_MATURITIES = [1, 2, 3, 4, 5]
FB_FORWARD_MATURITIES = [2, 3, 4, 5]

HORIZON_MIN_TRAIN = 60
HORIZON_RIDGE = 1.0
CP_FIRST_STAGE_RIDGE = 1e-10

# Primary comparison cost. Existing cost-sensitivity machinery is reused below.
V08_COST_BPS = BASE_COST_BPS

print("Configuration loaded.")


## 2. Data download and caching

In [ ]:
def _fred_cache_path(name: str) -> Path:
    return CACHE_DIR / f"fred_{name}.csv"


def download_fred_series(
    series_map: dict,
    start: str,
    end=None,
    refresh: bool = False,
) -> pd.DataFrame:
    cache_name = "__".join(series_map.values())
    cache = _fred_cache_path(cache_name)

    if cache.exists() and not refresh:
        out = pd.read_csv(
            cache,
            index_col=0,
            parse_dates=True,
        )
        out.index.name = "Date"
        return out

    ids = list(series_map.values())

    raw = pdr.DataReader(
        ids,
        "fred",
        start=start,
        end=end,
    )

    inverse = {
        v: k
        for k, v in series_map.items()
    }

    raw = (
        raw.rename(columns=inverse)
        .sort_index()
    )

    raw.to_csv(cache)

    return raw


## 3. Download real U.S. zero-coupon curve and macro state

In [ ]:
zero_map = {
    f"{m}Y": fred_id
    for m, fred_id in ZERO_SERIES.items()
}

zero_daily_pct = download_fred_series(
    zero_map,
    start=DATA_START,
    end=DATA_END,
    refresh=REFRESH_DATA,
)

state_raw = download_fred_series(
    STATE_SERIES,
    start=DATA_START,
    end=DATA_END,
    refresh=REFRESH_DATA,
)

display(zero_daily_pct.tail())
display(state_raw.tail())


## 4. Monthly fitted zero curve

In [ ]:
# FRED yields are in percent.
zero_daily = zero_daily_pct / 100.0

zero_monthly = (
    zero_daily
    .resample("ME")
    .last()
    .dropna(how="any")
)

print(
    "Curve sample:",
    zero_monthly.index.min().date(),
    "to",
    zero_monthly.index.max().date(),
    "| months =",
    len(zero_monthly),
)

display(zero_monthly.tail())


## 5. Zero-coupon pricing and exact carry/roll convention

For a continuously compounded zero yield \(y_t(\tau)\),

\[
P_t(\tau)=e^{-y_t(\tau)\tau}.
\]

The realized one-month holding-period return is obtained by repricing the remaining maturity \(\tau-h\) on next month's fitted curve.

The **carry/roll** component is the return that would occur if the *current* fitted curve remained unchanged while the bond aged by one month:

\[
R^{CR}_{t\rightarrow t+h}(\tau)
=
\frac{
P_t(\tau-h)
}{
P_t(\tau)
}
-1.
\]

The residual repricing component is defined additively as:

\[
R^{curve}
=
R^{realized}
-
R^{CR}.
\]

This additive decomposition is exact by definition for the research P&L attribution used here. It is not claimed to be the unique economic carry/roll convention.


In [ ]:
def interp_curve_yield(
    curve_row: pd.Series,
    maturity: float,
) -> float:
    tenors = np.array(
        [
            float(c.replace("Y", ""))
            for c in curve_row.index
        ]
    )

    yields = curve_row.to_numpy(
        dtype=float
    )

    if maturity < tenors.min() or maturity > tenors.max():
        raise ValueError(
            f"Maturity {maturity:.4f} outside curve grid."
        )

    return float(
        np.interp(
            maturity,
            tenors,
            yields,
        )
    )


def zero_coupon_price_cc(
    y: float,
    maturity: float,
) -> float:
    return float(
        np.exp(-y * maturity)
    )


def build_zero_coupon_return_decomposition(
    zero_curve: pd.DataFrame,
    maturities: list[int],
    horizon_years: float = 1.0 / 12.0,
):
    realized_rows = []
    carry_rows = []
    curve_rows = []

    for i in range(
        len(zero_curve) - 1
    ):
        dt0 = zero_curve.index[i]
        dt1 = zero_curve.index[i + 1]

        curve0 = zero_curve.iloc[i]
        curve1 = zero_curve.iloc[i + 1]

        realized = {}
        carry_roll = {}
        curve_component = {}

        for tau in maturities:
            asset = f"ZC{tau}Y"
            remaining = tau - horizon_years

            y0_tau = float(
                curve0[f"{tau}Y"]
            )

            y0_remaining = interp_curve_yield(
                curve0,
                remaining,
            )

            y1_remaining = interp_curve_yield(
                curve1,
                remaining,
            )

            p0 = zero_coupon_price_cc(
                y0_tau,
                tau,
            )

            p1_realized = zero_coupon_price_cc(
                y1_remaining,
                remaining,
            )

            p1_unchanged_curve = zero_coupon_price_cc(
                y0_remaining,
                remaining,
            )

            r_realized = (
                p1_realized / p0 - 1.0
            )

            r_carry = (
                p1_unchanged_curve / p0 - 1.0
            )

            r_curve = (
                r_realized - r_carry
            )

            realized[asset] = r_realized
            carry_roll[asset] = r_carry
            curve_component[asset] = r_curve

        # Realized returns are dated at the END of the holding month.
        realized_rows.append(
            pd.Series(
                realized,
                name=dt1,
            )
        )

        curve_rows.append(
            pd.Series(
                curve_component,
                name=dt1,
            )
        )

        # Carry/roll is KNOWN at dt0 for the return over dt0 -> dt1.
        carry_rows.append(
            pd.Series(
                carry_roll,
                name=dt0,
            )
        )

    return (
        pd.DataFrame(realized_rows).sort_index(),
        pd.DataFrame(carry_rows).sort_index(),
        pd.DataFrame(curve_rows).sort_index(),
    )


zc_returns, zc_carry_forward, zc_curve_realized = (
    build_zero_coupon_return_decomposition(
        zero_monthly,
        TRADED_MATURITIES,
    )
)

display(zc_returns.tail())
display(zc_carry_forward.tail())


## 6. Cash return and excess-return diagnostics

In [ ]:
fedfunds_monthly = (
    state_raw["FEDFUNDS"]
    .resample("ME")
    .last()
    .reindex(zc_returns.index)
    .ffill()
)

cash_return = (
    fedfunds_monthly.shift(1)
    / 100.0
    / 12.0
).rename("Cash")

zc_excess = (
    zc_returns
    .sub(
        cash_return,
        axis=0,
    )
)

display(
    pd.concat(
        [
            zc_returns.mean().rename(
                "mean_total"
            ),
            zc_excess.mean().rename(
                "mean_excess"
            ),
        ],
        axis=1,
    )
)


## 7. Lagged EWMA risk model and volatility-standardized yield changes

In [ ]:
def ewma_covariance_path(
    x: pd.DataFrame,
    lam: float = 0.97,
    warmup: int = 36,
) -> dict[pd.Timestamp, pd.DataFrame]:
    clean = x.astype(float)
    cols = list(clean.columns)

    if len(clean) <= warmup:
        raise ValueError(
            "Not enough observations for EWMA warmup."
        )

    init = (
        clean.iloc[:warmup]
        .dropna(how="any")
    )

    cov = np.cov(
        init.to_numpy(),
        rowvar=False,
        ddof=1,
    )

    out = {}

    for i, dt in enumerate(
        clean.index
    ):
        if i >= warmup:
            out[dt] = pd.DataFrame(
                cov.copy(),
                index=cols,
                columns=cols,
            )

            shock = clean.iloc[
                i
            ].to_numpy(dtype=float)

            if np.all(
                np.isfinite(shock)
            ):
                cov = (
                    lam * cov
                    + (1.0 - lam)
                    * np.outer(
                        shock,
                        shock,
                    )
                )

    return out


def covariance_panel_to_vols(
    cov_path,
):
    rows = []

    for dt, cov in cov_path.items():
        sigma = np.sqrt(
            np.clip(
                np.diag(
                    cov.to_numpy(
                        dtype=float
                    )
                ),
                0.0,
                None,
            )
        )

        rows.append(
            pd.Series(
                sigma,
                index=cov.index,
                name=dt,
            )
        )

    return pd.DataFrame(
        rows
    ).sort_index()


def standardize_with_lagged_vol(
    dy: pd.DataFrame,
    cov_path,
):
    rows = []

    for dt in dy.index:
        if dt not in cov_path:
            continue

        cov = cov_path[dt]

        sigma = np.sqrt(
            np.clip(
                np.diag(
                    cov.to_numpy(
                        dtype=float
                    )
                ),
                1e-12,
                None,
            )
        )

        z = (
            dy.loc[
                dt,
                cov.columns,
            ].to_numpy(
                dtype=float
            )
            / sigma
        )

        rows.append(
            pd.Series(
                z,
                index=cov.columns,
                name=dt,
            )
        )

    return pd.DataFrame(
        rows
    ).sort_index()


dy = (
    zero_monthly
    .diff()
    .dropna()
)

raw_cov_path = ewma_covariance_path(
    dy,
    lam=EWMA_LAMBDA,
    warmup=EWMA_WARMUP_MONTHS,
)

lagged_yield_vol = (
    covariance_panel_to_vols(
        raw_cov_path
    )
)

z_dy = (
    standardize_with_lagged_vol(
        dy,
        raw_cov_path,
    )
)

display(z_dy.tail())


## 8. Standardized PCA: Level / Slope / Curvature

In [ ]:
def _curvature_template(
    n: int,
) -> np.ndarray:
    x = np.linspace(
        -1.0,
        1.0,
        n,
    )

    t = (
        1.0
        - 2.0 * x**2
    )

    t = t - t.mean()

    nrm = np.linalg.norm(t)

    return (
        t / nrm
        if nrm > 0
        else t
    )


@dataclass
class CurvePCA:
    n_components: int = 3

    def fit(
        self,
        z: pd.DataFrame,
    ):
        clean = z.dropna(
            how="any"
        )

        self.tenors_ = list(
            clean.columns
        )

        self.mean_ = clean.mean(
            axis=0
        )

        x = (
            clean.to_numpy(
                dtype=float
            )
            - self.mean_.to_numpy(
                dtype=float
            )
        )

        cov = np.cov(
            x,
            rowvar=False,
            ddof=1,
        )

        eigvals, eigvecs = (
            np.linalg.eigh(cov)
        )

        order = np.argsort(
            eigvals
        )[::-1]

        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]

        loadings = eigvecs[
            :,
            : self.n_components,
        ].copy()

        if loadings[
            :, 0
        ].sum() < 0:
            loadings[
                :, 0
            ] *= -1.0

        if self.n_components >= 2:
            slope_template = np.linspace(
                -1.0,
                1.0,
                loadings.shape[0],
            )

            if np.dot(
                loadings[:, 1],
                slope_template,
            ) < 0:
                loadings[
                    :, 1
                ] *= -1.0

        if self.n_components >= 3:
            if np.dot(
                loadings[:, 2],
                _curvature_template(
                    loadings.shape[0]
                ),
            ) < 0:
                loadings[
                    :, 2
                ] *= -1.0

        self.factor_names_ = [
            "Level",
            "Slope",
            "Curvature",
        ][: self.n_components]

        self.eigenvalues_ = (
            eigvals[
                : self.n_components
            ]
        )

        self.explained_variance_ratio_ = (
            eigvals[
                : self.n_components
            ]
            / eigvals.sum()
        )

        self.loadings_ = pd.DataFrame(
            loadings,
            index=self.tenors_,
            columns=self.factor_names_,
        )

        return self

    def transform(
        self,
        z: pd.DataFrame,
    ) -> pd.DataFrame:
        x = (
            z.loc[
                :,
                self.tenors_,
            ]
            .subtract(
                self.mean_,
                axis=1,
            )
        )

        scores = (
            x.to_numpy(
                dtype=float
            )
            @ self.loadings_.to_numpy(
                dtype=float
            )
        )

        return pd.DataFrame(
            scores,
            index=z.index,
            columns=self.factor_names_,
        )

    def diagnostics(self):
        return pd.DataFrame(
            {
                "eigenvalue":
                    self.eigenvalues_,
                "explained_variance_ratio":
                    self.explained_variance_ratio_,
            },
            index=self.factor_names_,
        )


pca_train = (
    z_dy.loc[
        :PCA_TRAIN_END
    ]
)

if len(pca_train) < 36:
    raise RuntimeError(
        "PCA training sample too short."
    )

pca = CurvePCA(
    n_components=N_FACTORS
).fit(
    pca_train
)

factor_scores = (
    pca.transform(
        z_dy
    )
)

factor_scores_1sigma = (
    factor_scores
    / pd.Series(
        np.sqrt(
            pca.eigenvalues_
        ),
        index=FACTOR_NAMES,
    )
)

display(
    pca.loadings_.round(4)
)

display(
    pca.diagnostics().round(4)
)


In [ ]:
ax = pca.loadings_.plot(
    marker="o",
    figsize=(10, 5),
)

ax.set_title(
    "Volatility-standardized PCA loadings"
)

ax.set_xlabel(
    "Maturity"
)

ax.set_ylabel(
    "Loading"
)

plt.show()


## 9. Time-varying one-sigma factor shocks in raw yield units

In [ ]:
def factor_shocks_decimal_at(
    pca: CurvePCA,
    raw_cov: pd.DataFrame,
) -> pd.DataFrame:
    sigma = np.sqrt(
        np.clip(
            np.diag(
                raw_cov.to_numpy(
                    dtype=float
                )
            ),
            0.0,
            None,
        )
    )

    D = np.diag(
        sigma
    )

    B = pca.loadings_.to_numpy(
        dtype=float
    )

    scale = np.sqrt(
        pca.eigenvalues_
    )

    shocks = (
        D
        @ B
        @ np.diag(scale)
    )

    return pd.DataFrame(
        shocks,
        index=pca.tenors_,
        columns=pca.factor_names_,
    )


latest_shocks = (
    factor_shocks_decimal_at(
        pca,
        raw_cov_path[
            z_dy.index[-1]
        ],
    )
)

print(
    "Latest one-sigma factor shocks (bp)"
)

display(
    (
        latest_shocks
        * 10000.0
    ).round(2)
)


## 10. Analytic zero-coupon factor exposures

For a zero-coupon bond, the first-order return response to a yield shock is approximately

\[
\Delta R
\approx
-\tau \Delta y.
\]

Hence the return beta to each current one-sigma curve factor is available directly from maturity and the mapped factor shock.


In [ ]:
def build_zero_coupon_factor_betas(
    dates,
    maturities,
    pca,
    cov_path,
):
    assets = [
        f"ZC{m}Y"
        for m in maturities
    ]

    out = {
        factor: pd.DataFrame(
            index=dates,
            columns=assets,
            dtype=float,
        )
        for factor in FACTOR_NAMES
    }

    for dt in dates:
        if dt not in cov_path:
            continue

        shocks = factor_shocks_decimal_at(
            pca,
            cov_path[dt],
        )

        for m in maturities:
            asset = f"ZC{m}Y"

            for factor in FACTOR_NAMES:
                shock = float(
                    shocks.loc[
                        f"{m}Y",
                        factor,
                    ]
                )

                out[
                    factor
                ].loc[
                    dt,
                    asset,
                ] = (
                    -float(m)
                    * shock
                )

    return out


zc_betas_current = (
    build_zero_coupon_factor_betas(
        dates=zc_carry_forward.index,
        maturities=TRADED_MATURITIES,
        pca=pca,
        cov_path=raw_cov_path,
    )
)

display(
    zc_betas_current[
        "Level"
    ].tail().round(4)
)


## 11. Funded exact factor-mimicking portfolios

The v0.5.1 portfolios imposed both:

\[
\widetilde B_t'w=e_k
\]

and

\[
\mathbf 1'w=0.
\]

That second condition is unnecessary for a self-financing overlay because cash can fund the risky bond sleeve.

v0.6 solves:

\[
\min_w \frac12w'H_tw
\]

subject only to

\[
\widetilde B_t'w=e_k.
\]

The bond portfolio is then gross-normalized. Cross-factor neutrality is preserved exactly.

The funding leg is:

\[
w^{cash}=-\mathbf 1'w.
\]

Hence the factor excess return is simply:

\[
R^{factor}=w'R^{excess}.
\]

This lets the Level factor retain genuine funded duration exposure rather than forcing it into a maturity-relative-value construction.


In [ ]:
def factor_beta_matrix_at(
    dt,
    betas,
) -> pd.DataFrame:
    assets = next(iter(betas.values())).columns

    return pd.DataFrame(
        {
            factor: betas[factor].loc[dt, assets]
            for factor in FACTOR_NAMES
        },
        index=assets,
    ).astype(float)


def rms_standardize_columns(
    B: pd.DataFrame,
):
    scale = np.sqrt((B ** 2).mean(axis=0))
    scale = scale.replace(0.0, np.nan)
    return B.divide(scale, axis=1), scale


def trailing_objective_covariance(
    asset_returns: pd.DataFrame,
    dt: pd.Timestamp,
    assets,
    lookback: int,
    min_obs: int,
    diag_shrink: float,
    numerical_ridge: float,
) -> pd.DataFrame:
    hist = (
        asset_returns
        .loc[:dt, assets]
        .tail(lookback)
        .dropna(how="any")
    )

    if len(hist) < min_obs:
        return pd.DataFrame(
            np.eye(len(assets)),
            index=assets,
            columns=assets,
        )

    sample = hist.cov().to_numpy(dtype=float)
    diagonal = np.diag(np.diag(sample))

    H = (
        (1.0 - diag_shrink) * sample
        + diag_shrink * diagonal
    )

    diag_mean = float(np.mean(np.diag(H)))
    ridge = numerical_ridge * max(diag_mean, 1e-12)

    H = H + ridge * np.eye(H.shape[0])

    return pd.DataFrame(
        H,
        index=assets,
        columns=assets,
    )


def exact_funded_factor_weights(
    B: pd.DataFrame,
    H: pd.DataFrame,
    target_factor: str,
    target_gross: float = 1.0,
):
    """
    Solve:
        min 0.5 * w' H w
        s.t. Bn' w = e_k

    No risky-asset net-zero constraint.
    Cash funding is added explicitly later.
    """
    Bn, beta_scale = rms_standardize_columns(B)

    if Bn.isna().any().any():
        raise ValueError("Invalid factor beta matrix.")

    assets = list(Bn.index)

    C = Bn.to_numpy(dtype=float)

    q = np.zeros(len(FACTOR_NAMES))
    q[FACTOR_NAMES.index(target_factor)] = 1.0

    Hm = (
        H.reindex(index=assets, columns=assets)
        .to_numpy(dtype=float)
    )

    H_inv_C = np.linalg.solve(Hm, C)

    middle = C.T @ H_inv_C

    w_raw = (
        H_inv_C
        @ np.linalg.pinv(middle)
        @ q
    )

    raw_gross = float(np.abs(w_raw).sum())

    if raw_gross <= 1e-14:
        raise RuntimeError("Degenerate funded factor portfolio.")

    # Gross-normalization preserves exact zero cross-factor constraints.
    w = (
        w_raw
        * target_gross
        / raw_gross
    )

    w = pd.Series(
        w,
        index=assets,
        name=target_factor,
    )

    achieved_norm = Bn.T @ w
    achieved_raw = B.T @ w

    target_loading = float(
        achieved_norm[target_factor]
    )

    cross = achieved_norm.drop(
        target_factor
    )

    total_factor_abs = float(
        achieved_norm.abs().sum()
    )

    purity = (
        abs(target_loading) / total_factor_abs
        if total_factor_abs > 0
        else np.nan
    )

    risky_net = float(w.sum())
    cash_funding_weight = -risky_net

    diagnostics = pd.Series(
        {
            "risky_net_weight": risky_net,
            "cash_funding_weight": cash_funding_weight,
            "gross_risky_weight": float(w.abs().sum()),
            "purity_normalized_beta": purity,
            "target_factor_loading_norm": target_loading,
            "cross_factor_loading_norm": float(cross.abs().sum()),
            "max_cross_constraint_violation": float(
                cross.abs().max()
            ),
            "raw_solution_gross": raw_gross,
            "objective_variance": float(
                w.to_numpy() @ Hm @ w.to_numpy()
            ),
        },
        name=target_factor,
    )

    return w, achieved_raw, diagnostics, beta_scale


def build_funded_mimicking_weights_at(
    dt,
    B,
    asset_excess_returns,
):
    assets = list(B.index)

    H = trailing_objective_covariance(
        asset_returns=asset_excess_returns,
        dt=dt,
        assets=assets,
        lookback=MIMIC_COV_LOOKBACK,
        min_obs=MIMIC_COV_MIN_OBS,
        diag_shrink=MIMIC_COV_DIAG_SHRINK,
        numerical_ridge=MIMIC_NUMERICAL_RIDGE,
    )

    weights = {}
    achieved = {}
    diagnostics = {}

    for factor in FACTOR_NAMES:
        w, a, d, _ = exact_funded_factor_weights(
            B=B,
            H=H,
            target_factor=factor,
            target_gross=MIMIC_GROSS,
        )

        weights[factor] = w
        achieved[factor] = a
        diagnostics[factor] = d

    return (
        pd.DataFrame(weights),
        pd.DataFrame(achieved),
        pd.DataFrame(diagnostics).T,
        H,
    )


sample_dt = (
    zc_carry_forward.index
    .intersection(
        zc_betas_current["Level"]
        .dropna(how="all")
        .index
    )[-1]
)

sample_B = factor_beta_matrix_at(
    sample_dt,
    zc_betas_current,
)

sample_mimic_w, sample_achieved, sample_diag, sample_H = (
    build_funded_mimicking_weights_at(
        dt=sample_dt,
        B=sample_B,
        asset_excess_returns=zc_excess,
    )
)

print("Latest funded mimicking weights")
display(sample_mimic_w.round(3))

print("Latest funded factor diagnostics")
display(sample_diag.round(6))


### Solver self-check

The funded solver must enforce exact cross-factor neutrality while allowing a non-zero risky net position financed by cash.


In [ ]:
assert callable(build_funded_mimicking_weights_at)
assert callable(exact_funded_factor_weights)

print("Funded exact factor-portfolio solver loaded correctly.")


## 12. Build the full time series of factor-mimicking weights

In [ ]:
def build_mimicking_weight_panel(
    dates,
    betas,
    asset_excess_returns,
):
    assets = next(iter(betas.values())).columns

    weights = {
        factor: pd.DataFrame(
            index=dates,
            columns=assets,
            dtype=float,
        )
        for factor in FACTOR_NAMES
    }

    diag_rows = []

    for dt in dates:
        if any(
            dt not in betas[factor].index
            for factor in FACTOR_NAMES
        ):
            continue

        B = factor_beta_matrix_at(
            dt,
            betas,
        )

        if B.isna().any().any():
            continue

        w, achieved, diag, H = (
            build_funded_mimicking_weights_at(
                dt=dt,
                B=B,
                asset_excess_returns=asset_excess_returns,
            )
        )

        for factor in FACTOR_NAMES:
            weights[factor].loc[
                dt,
                w.index,
            ] = w[factor]

            row = diag.loc[factor].copy()
            row["date"] = dt
            row["factor"] = factor

            diag_rows.append(row)

    diagnostics = (
        pd.DataFrame(diag_rows)
        .set_index(["date", "factor"])
        .sort_index()
    )

    return weights, diagnostics


mimic_weights, mimic_purity = (
    build_mimicking_weight_panel(
        dates=zc_carry_forward.index,
        betas=zc_betas_current,
        asset_excess_returns=zc_excess,
    )
)

print("Average funded factor diagnostics")
display(
    mimic_purity
    .groupby(level="factor")
    .mean()
    .round(6)
)

print("Worst cross-factor loading")
print(
    float(
        mimic_purity[
            "cross_factor_loading_norm"
        ].max()
    )
)


## 12A. Funded-factor constraint validation

The required conditions are now:

\[
\beta_j'w_k \approx 0
\quad
\forall j\neq k
\]

and factor purity ≈ 1.

A non-zero risky net position is **allowed** and must be matched by:

\[
w^{cash}=-\sum_iw_i^{bond}.
\]

The backtest stops only if cross-factor neutrality or purity fail.


In [ ]:
MAX_CROSS_TOL = 1e-8

worst_cross = float(
    mimic_purity[
        "cross_factor_loading_norm"
    ].max()
)

median_purity = float(
    mimic_purity[
        "purity_normalized_beta"
    ].median()
)

funding_identity_error = float(
    (
        mimic_purity[
            "risky_net_weight"
        ]
        + mimic_purity[
            "cash_funding_weight"
        ]
    )
    .abs()
    .max()
)

constraint_check = pd.Series(
    {
        "worst_cross_factor_loading": worst_cross,
        "median_factor_purity": median_purity,
        "worst_funding_identity_error": funding_identity_error,
        "cross_factor_constraint_pass": worst_cross < MAX_CROSS_TOL,
        "purity_pass": median_purity > 0.999,
        "funding_identity_pass": funding_identity_error < 1e-12,
    },
    name="funded_factor_constraints",
)

display(constraint_check)

if not (
    constraint_check[
        "cross_factor_constraint_pass"
    ]
    and constraint_check[
        "purity_pass"
    ]
    and constraint_check[
        "funding_identity_pass"
    ]
):
    raise RuntimeError(
        "Funded factor-portfolio constraints failed."
    )


## 13. Realized factor portfolio returns and carry/roll decomposition

## 13. Funded factor excess returns and carry/roll decomposition

For factor weights \(w_t\), the explicit cash funding leg is:

\[
w_t^{cash}=-\mathbf1'w_t.
\]

Hence the realized self-financing factor excess return is:

\[
R^{factor}_{t+1}
=
w_t'
\left(
R^{bond}_{t+1}
-
R^f_{t+1}\mathbf1
\right).
\]

The known funded carry/roll at time \(t\) is:

\[
CR^{factor}_{t}
=
w_t'
\left(
CR^{bond}_{t}
-
R^f_{t\rightarrow t+1}\mathbf1
\right).
\]

Then:

\[
Curve^{factor}_{t+1}
=
R^{factor}_{t+1}
-
CR^{factor}_{t}.
\]

This is the economically consistent funded decomposition.


In [ ]:
# Cash return over t -> t+1, indexed by forecast date t.
cash_forward = (
    cash_return
    .shift(-1)
    .reindex(
        zc_carry_forward.index
    )
)


def build_factor_return_decomposition(
    mimic_weights,
    asset_excess_returns,
    carry_forward,
    cash_forward,
):
    factor_total = {}
    factor_curve = {}
    factor_carry_forward = {}

    for factor in FACTOR_NAMES:
        w_panel = (
            mimic_weights[factor]
            .dropna(how="all")
        )

        total_series = {}
        curve_series = {}
        carry_series = {}

        for dt in w_panel.index:
            if (
                dt not in carry_forward.index
                or dt not in cash_forward.index
                or pd.isna(cash_forward.loc[dt])
            ):
                continue

            future_dates = (
                asset_excess_returns.index[
                    asset_excess_returns.index > dt
                ]
            )

            if len(future_dates) == 0:
                continue

            next_dt = future_dates[0]

            w = (
                w_panel.loc[dt]
                .dropna()
            )

            assets = w.index

            # Explicitly funded factor return = w' excess bond return.
            realized_excess = float(
                w
                @ asset_excess_returns.loc[
                    next_dt,
                    assets,
                ]
            )

            funded_carry_vector = (
                carry_forward.loc[
                    dt,
                    assets,
                ]
                - float(
                    cash_forward.loc[dt]
                )
            )

            funded_carry = float(
                w
                @ funded_carry_vector
            )

            total_series[next_dt] = realized_excess
            curve_series[next_dt] = (
                realized_excess
                - funded_carry
            )
            carry_series[dt] = funded_carry

        factor_total[factor] = pd.Series(
            total_series,
            name=factor,
        )

        factor_curve[factor] = pd.Series(
            curve_series,
            name=factor,
        )

        factor_carry_forward[factor] = pd.Series(
            carry_series,
            name=factor,
        )

    return (
        pd.DataFrame(factor_total).sort_index(),
        pd.DataFrame(factor_carry_forward).sort_index(),
        pd.DataFrame(factor_curve).sort_index(),
    )


factor_total_returns, factor_carry_forward, factor_curve_realized = (
    build_factor_return_decomposition(
        mimic_weights=mimic_weights,
        asset_excess_returns=zc_excess,
        carry_forward=zc_carry_forward,
        cash_forward=cash_forward,
    )
)

print("Funded factor excess returns")
display(
    factor_total_returns.describe().T.round(4)
)

print("Known funded factor carry/roll")
display(
    factor_carry_forward.describe().T.round(4)
)

print("Realized funded curve component")
display(
    factor_curve_realized.describe().T.round(4)
)


In [ ]:
ax = (
    1.0
    + factor_total_returns.fillna(
        0.0
    )
).cumprod().plot(
    figsize=(11, 6)
)

ax.set_title(
    "Cumulative gross factor-mimicking portfolio returns"
)

ax.set_ylabel(
    "Growth of $1 overlay notional"
)

plt.show()


## 14. Factor-return stability and correlation diagnostics

In [ ]:
factor_diag = pd.DataFrame(
    {
        "mean_monthly":
            factor_total_returns.mean(),
        "ann_vol":
            factor_total_returns.std(
                ddof=1
            )
            * np.sqrt(12.0),
        "carry_mean":
            factor_carry_forward.mean(),
        "curve_mean":
            factor_curve_realized.mean(),
    }
)

display(
    factor_diag.round(5)
)

print("Factor-return correlation")
display(
    factor_total_returns.corr().round(3)
)


## 15. Fama–MacBeth retained only as a diagnostic

In [ ]:
if RUN_FMB_DIAGNOSTIC:

    @dataclass
    class FamaMacBethEstimator:
        ridge: float = 1e-8

        def fit(
            self,
            returns,
            betas,
        ):
            factors = list(
                betas
            )

            common_dates = (
                returns.index
            )

            common_assets = (
                returns.columns
            )

            for b in betas.values():
                common_dates = (
                    common_dates
                    .intersection(
                        b.index
                    )
                )

                common_assets = (
                    common_assets
                    .intersection(
                        b.columns
                    )
                )

            rows = []
            diag_rows = []

            for dt in common_dates:
                y = (
                    returns.loc[
                        dt,
                        common_assets,
                    ]
                )

                X = pd.DataFrame(
                    {
                        f:
                            betas[
                                f
                            ].loc[
                                dt,
                                common_assets,
                            ]
                        for f in factors
                    }
                )

                frame = (
                    pd.concat(
                        [
                            y.rename("y"),
                            X,
                        ],
                        axis=1,
                    )
                    .dropna()
                )

                if len(frame) < 7:
                    continue

                yv = frame[
                    "y"
                ].to_numpy(
                    dtype=float
                )

                xv = frame[
                    factors
                ].to_numpy(
                    dtype=float
                )

                xv = np.column_stack(
                    [
                        np.ones(
                            len(xv)
                        ),
                        xv,
                    ]
                )

                gram = (
                    xv.T
                    @ xv
                )

                penalty = (
                    np.eye(
                        gram.shape[0]
                    )
                    * self.ridge
                )

                penalty[
                    0, 0
                ] = 0.0

                coef = (
                    np.linalg.pinv(
                        gram
                        + penalty
                    )
                    @ (
                        xv.T
                        @ yv
                    )
                )

                rows.append(
                    pd.Series(
                        coef,
                        index=[
                            "alpha",
                            *factors,
                        ],
                        name=dt,
                    )
                )

                diag_rows.append(
                    (
                        dt,
                        float(
                            np.linalg.cond(
                                xv
                            )
                        ),
                    )
                )

            self.coef_ = (
                pd.DataFrame(
                    rows
                )
                .sort_index()
            )

            self.diagnostics_ = (
                pd.DataFrame(
                    diag_rows,
                    columns=[
                        "date",
                        "condition_number",
                    ],
                )
                .set_index(
                    "date"
                )
            )

            return self

    # Return at t uses exposure known at t-1.
    diagnostic_betas = {
        factor:
            zc_betas_current[
                factor
            ].shift(
                1
            )
        for factor in FACTOR_NAMES
    }

    fm_diag = (
        FamaMacBethEstimator()
        .fit(
            returns=zc_excess,
            betas=diagnostic_betas,
        )
    )

    print(
        "FMB condition number remains a diagnostic, not the main signal engine."
    )

    display(
        fm_diag.diagnostics_.describe().round(1)
    )


## 16. Observable economic and market state

In [ ]:
def rolling_zscore(
    x: pd.DataFrame,
    window: int = 60,
    min_periods: int = 36,
):
    mean = (
        x.rolling(
            window,
            min_periods=min_periods,
        )
        .mean()
    )

    std = (
        x.rolling(
            window,
            min_periods=min_periods,
        )
        .std(
            ddof=1
        )
    )

    return (
        x - mean
    ) / std


curve_state = pd.DataFrame(
    index=zero_monthly.index
)

curve_state[
    "curve_level"
] = zero_monthly.mean(
    axis=1
)

curve_state[
    "curve_slope_10_2"
] = (
    zero_monthly["10Y"]
    - zero_monthly["2Y"]
)

curve_state[
    "curve_curvature"
] = (
    2.0 * zero_monthly["5Y"]
    - zero_monthly["2Y"]
    - zero_monthly["10Y"]
)

curve_state[
    "rate_vol"
] = (
    lagged_yield_vol
    .mean(
        axis=1
    )
)

# PCA scores are observable market-state variables.
for factor in FACTOR_NAMES:
    curve_state[
        f"pc_{factor.lower()}"
    ] = factor_scores_1sigma[
        factor
    ]

macro = pd.DataFrame()

macro[
    "inflation_yoy"
] = (
    state_raw["CPI"]
    .resample("ME")
    .last()
    .pct_change(12)
    .shift(1)
)

macro[
    "growth_yoy"
] = (
    state_raw["INDPRO"]
    .resample("ME")
    .last()
    .pct_change(12)
    .shift(1)
)

macro[
    "fedfunds"
] = (
    state_raw["FEDFUNDS"]
    .resample("ME")
    .last()
    .shift(1)
    / 100.0
)

macro[
    "breakeven10y"
] = (
    state_raw[
        "BREAKEVEN10Y"
    ]
    .resample("ME")
    .last()
    / 100.0
)

state = (
    curve_state
    .join(
        macro,
        how="outer",
    )
    .sort_index()
)

state = (
    rolling_zscore(
        state,
        window=60,
        min_periods=36,
    )
    .dropna(
        how="any"
    )
)

display(
    state.tail().round(3)
)


## 17. Conditional forecast of future curve-repricing factor returns

## 17. Conditional forecast of future curve-repricing factor returns

The regression still forecasts the future curve-repricing component:

\[
CurveRet_{k,t+1}
=
\alpha_k+\gamma_k'z_t+\varepsilon_{k,t+1}.
\]

But v0.7 retains **two different uncertainty objects**.

### 1. Mean-estimation uncertainty

\[
SE_t(\hat\mu_k)
\]

measures uncertainty about the estimated conditional mean.

This is the relevant object for deciding whether the expected premium is credibly positive or negative.

### 2. Return residual volatility

\[
\sigma_{\varepsilon,k}
\]

measures the dispersion of the next realized factor return around that mean.

This is relevant to realized investment risk, but it should not be used as if it were uncertainty about whether the expected premium exists.

For ridge regression,

\[
\hat\beta
=
(X'X+\Lambda)^{-1}X'y,
\]

and the approximate coefficient covariance used here is

\[
\widehat{Var}(\hat\beta)
=
\hat\sigma^2
A^{-1}X'XA^{-1},
\qquad
A=X'X+\Lambda.
\]

Thus

\[
SE_t(\hat\mu)
=
\sqrt{
x_t'
\widehat{Var}(\hat\beta)
x_t
}.
\]

Carry/roll is known at \(t\), so it shifts the expected-return mean without adding estimation uncertainty.


In [ ]:
@dataclass
class ConditionalFactorModel:
    horizon: int = 1
    ridge: float = 1.0
    add_intercept: bool = True

    def fit(
        self,
        target_returns: pd.DataFrame,
        state: pd.DataFrame,
    ):
        common = (
            target_returns.index
            .intersection(state.index)
            .sort_values()
        )

        y_panel = target_returns.loc[common]
        z = state.loc[common]

        target = y_panel.shift(
            -self.horizon
        )

        self.feature_names_ = list(
            z.columns
        )

        self.factor_names_ = list(
            y_panel.columns
        )

        coef = {}
        sigma = {}
        coef_cov = {}

        for factor in self.factor_names_:
            frame = (
                pd.concat(
                    [
                        target[
                            factor
                        ].rename("y"),
                        z,
                    ],
                    axis=1,
                )
                .dropna()
            )

            y = frame[
                "y"
            ].to_numpy(
                dtype=float
            )

            X = frame[
                self.feature_names_
            ].to_numpy(
                dtype=float
            )

            if self.add_intercept:
                X = np.column_stack(
                    [
                        np.ones(len(X)),
                        X,
                    ]
                )

                names = [
                    "intercept",
                    *self.feature_names_,
                ]
            else:
                names = self.feature_names_

            XtX = X.T @ X

            penalty = (
                np.eye(
                    XtX.shape[0]
                )
                * self.ridge
            )

            if self.add_intercept:
                penalty[0, 0] = 0.0

            A = XtX + penalty
            A_inv = np.linalg.pinv(A)

            b = (
                A_inv
                @ X.T
                @ y
            )

            resid = y - X @ b

            dof = max(
                len(y) - len(b),
                1,
            )

            s2 = (
                float(
                    resid @ resid
                )
                / dof
            )

            s = sqrt(s2)

            # Approximate frequentist covariance for ridge coefficients.
            Vb = (
                s2
                * A_inv
                @ XtX
                @ A_inv
            )

            coef[factor] = pd.Series(
                b,
                index=names,
            )

            sigma[factor] = s
            coef_cov[factor] = pd.DataFrame(
                Vb,
                index=names,
                columns=names,
            )

        self.coef_ = pd.DataFrame(
            coef
        )

        self.residual_sigma_ = pd.Series(
            sigma,
            name="curve_sigma",
        )

        self.coef_cov_ = coef_cov

        return self

    def predict(
        self,
        state_row: pd.Series,
    ) -> pd.DataFrame:
        z = state_row.reindex(
            self.feature_names_
        )

        if z.isna().any():
            raise ValueError(
                "Missing state feature."
            )

        if self.add_intercept:
            x = np.r_[
                1.0,
                z.to_numpy(
                    dtype=float
                ),
            ]
        else:
            x = z.to_numpy(
                dtype=float
            )

        mu = pd.Series(
            x
            @ self.coef_.to_numpy(
                dtype=float
            ),
            index=self.factor_names_,
            name="curve_mean",
        )

        mean_se = {}

        for factor in self.factor_names_:
            Vb = self.coef_cov_[
                factor
            ].to_numpy(
                dtype=float
            )

            v = float(
                x
                @ Vb
                @ x
            )

            mean_se[factor] = sqrt(
                max(v, 1e-18)
            )

        mean_se = pd.Series(
            mean_se,
            name="curve_mean_se",
        )

        return pd.DataFrame(
            {
                "curve_mean": mu,
                "curve_mean_se": mean_se,
                "curve_sigma":
                    self.residual_sigma_,
            }
        )


def walk_forward_curve_forecasts(
    curve_returns,
    state,
    min_train,
    horizon=1,
    ridge=1.0,
    window=None,
):
    common = (
        curve_returns.index
        .intersection(state.index)
        .sort_values()
    )

    y = curve_returns.loc[
        common
    ]

    z = state.loc[
        common
    ]

    rows = []

    for i in range(
        min_train,
        len(common) - horizon,
    ):
        forecast_date = common[i]

        train_start = (
            0
            if window is None
            else max(
                0,
                i - window + 1,
            )
        )

        fit_idx = common[
            train_start :
            i + 1
        ]

        model = ConditionalFactorModel(
            horizon=horizon,
            ridge=ridge,
        )

        model.fit(
            y.loc[fit_idx],
            z.loc[fit_idx],
        )

        pred = model.predict(
            z.loc[
                forecast_date
            ]
        )

        pred[
            "forecast_date"
        ] = forecast_date

        pred[
            "factor"
        ] = pred.index

        rows.append(
            pred.reset_index(
                drop=True
            )
        )

    out = pd.concat(
        rows,
        ignore_index=True,
    )

    return (
        out.set_index(
            [
                "forecast_date",
                "factor",
            ]
        )
        .sort_index()
    )


conditional_curve_forecasts = (
    walk_forward_curve_forecasts(
        curve_returns=
            factor_curve_realized,
        state=state,
        min_train=COND_MIN_TRAIN,
        horizon=FORECAST_HORIZON,
        ridge=COND_RIDGE,
        window=COND_WINDOW,
    )
)

display(
    conditional_curve_forecasts.head(12)
)


## 18. Unconditional expanding curve-return baseline

In [ ]:
def expanding_curve_forecasts(
    curve_returns,
    forecast_dates,
    min_obs,
):
    rows = []

    for dt in forecast_dates:
        hist = (
            curve_returns.loc[:dt]
            .dropna()
        )

        if len(hist) < min_obs:
            continue

        mu = hist.mean()
        sig = hist.std(
            ddof=1
        )

        n = hist.count()

        mean_se = (
            sig
            / np.sqrt(n)
        )

        tmp = pd.DataFrame(
            {
                "curve_mean": mu,
                "curve_mean_se": mean_se,
                "curve_sigma": sig,
            }
        )

        tmp[
            "forecast_date"
        ] = dt

        tmp[
            "factor"
        ] = tmp.index

        rows.append(
            tmp.reset_index(
                drop=True
            )
        )

    out = pd.concat(
        rows,
        ignore_index=True,
    )

    return (
        out.set_index(
            [
                "forecast_date",
                "factor",
            ]
        )
        .sort_index()
    )


forecast_dates = (
    conditional_curve_forecasts
    .index
    .get_level_values(
        "forecast_date"
    )
    .unique()
)

unconditional_curve_forecasts = (
    expanding_curve_forecasts(
        curve_returns=
            factor_curve_realized,
        forecast_dates=
            forecast_dates,
        min_obs=
            COND_MIN_TRAIN,
    )
)


## 19. Combine known carry with predicted curve repricing

In [ ]:
def combine_carry_and_curve_forecast(
    curve_forecast_panel,
    factor_carry_forward,
):
    rows = []

    dates = (
        curve_forecast_panel
        .index
        .get_level_values(
            "forecast_date"
        )
        .unique()
        .intersection(
            factor_carry_forward.index
        )
    )

    for dt in dates:
        curve = (
            curve_forecast_panel
            .xs(
                dt,
                level="forecast_date",
            )
            .reindex(
                FACTOR_NAMES
            )
        )

        carry = (
            factor_carry_forward
            .loc[
                dt,
                FACTOR_NAMES,
            ]
        )

        total_mean = (
            carry
            + curve[
                "curve_mean"
            ]
        )

        out = pd.DataFrame(
            {
                "carry":
                    carry,
                "curve_mean":
                    curve[
                        "curve_mean"
                    ],
                "curve_mean_se":
                    curve[
                        "curve_mean_se"
                    ],
                "curve_sigma":
                    curve[
                        "curve_sigma"
                    ],
                "total_mean":
                    total_mean,
            }
        )

        out[
            "forecast_date"
        ] = dt

        out[
            "factor"
        ] = out.index

        rows.append(
            out.reset_index(
                drop=True
            )
        )

    return (
        pd.concat(
            rows,
            ignore_index=True,
        )
        .set_index(
            [
                "forecast_date",
                "factor",
            ]
        )
        .sort_index()
    )


conditional_total_forecasts = (
    combine_carry_and_curve_forecast(
        conditional_curve_forecasts,
        factor_carry_forward,
    )
)

unconditional_total_forecasts = (
    combine_carry_and_curve_forecast(
        unconditional_curve_forecasts,
        factor_carry_forward,
    )
)

display(
    conditional_total_forecasts.tail(9)
)


## 20. Mean-confidence factor signal

The signal gate now evaluates the **conditional expected return**, not the next realized return.

Let

\[
\hat\mu^{total}_{k,t}
=
Carry_{k,t}
+
\hat\mu^{curve}_{k,t}.
\]

Because carry is observed at \(t\), uncertainty comes only from estimation of the curve-repricing mean:

\[
SE_t(\hat\mu^{total})
=
SE_t(\hat\mu^{curve}).
\]

We evaluate:

\[
P(\mu^{total}_{k,t}>h)
\]

and

\[
P(\mu^{total}_{k,t}<-h)
\]

where the baseline hurdle \(h=0\). Trading costs are handled directly in realized performance rather than embedded through an arbitrary volatility-scaled return hurdle.

`curve_sigma` remains in the output as the volatility of realized curve returns, but it no longer determines whether the expected premium is statistically credible.


In [ ]:
def signed_factor_signal(
    forecast: pd.DataFrame,
    confidence_threshold: float,
    expected_return_hurdle: float = 0.0,
):
    mu = (
        forecast[
            "total_mean"
        ]
        .astype(float)
    )

    mean_se = (
        forecast[
            "curve_mean_se"
        ]
        .astype(float)
        .clip(
            lower=1e-12
        )
    )

    realized_sigma = (
        forecast[
            "curve_sigma"
        ]
        .astype(float)
        .clip(
            lower=1e-12
        )
    )

    hurdle = float(
        expected_return_hurdle
    )

    p_positive_mean = (
        1.0
        - norm.cdf(
            (
                hurdle
                - mu
            )
            / mean_se
        )
    )

    p_negative_mean = (
        norm.cdf(
            (
                -hurdle
                - mu
            )
            / mean_se
        )
    )

    p_positive_mean = pd.Series(
        p_positive_mean,
        index=mu.index,
    )

    p_negative_mean = pd.Series(
        p_negative_mean,
        index=mu.index,
    )

    side = pd.Series(
        0.0,
        index=mu.index,
    )

    side[
        (
            p_positive_mean
            >= confidence_threshold
        )
        & (
            p_positive_mean
            >= p_negative_mean
        )
    ] = 1.0

    side[
        (
            p_negative_mean
            >= confidence_threshold
        )
        & (
            p_negative_mean
            > p_positive_mean
        )
    ] = -1.0

    # Diagnostic only: probability of the next realized curve return sign.
    predictive_sigma = np.sqrt(
        realized_sigma**2
        + mean_se**2
    )

    p_next_positive = (
        1.0
        - norm.cdf(
            (
                0.0
                - mu
            )
            / predictive_sigma
        )
    )

    return pd.DataFrame(
        {
            "total_mean": mu,
            "mean_se": mean_se,
            "curve_sigma": realized_sigma,
            "p_positive_mean":
                p_positive_mean,
            "p_negative_mean":
                p_negative_mean,
            "p_next_positive_return":
                p_next_positive,
            "side": side,
        }
    )


## 21. Factor allocation

Factor allocation is deliberately simple:

\[
score_{k,t}
=
\frac{
E_t[R_{k,t+1}]
}{
\sigma^{factor}_{k,t}
}
\]

only when the probabilistic filter is active.

Scores keep their sign, are capped at the factor level, and are normalized to a total factor gross budget.

No parameter is optimized on realized P&L in this first run.


In [ ]:
def trailing_factor_vol(
    factor_returns,
    dt,
    lookback,
    min_obs,
):
    hist = (
        factor_returns.loc[
            :dt,
            FACTOR_NAMES,
        ]
        .tail(
            lookback
        )
    )

    vol = hist.std(
        ddof=1
    )

    count = hist.count()

    vol[
        count
        < min_obs
    ] = np.nan

    return vol


def capped_signed_normalize(
    raw_score: pd.Series,
    max_abs_weight: float,
    gross_target: float,
):
    score = raw_score.copy()

    if (
        score.abs().sum()
        <= 0
    ):
        return pd.Series(
            0.0,
            index=score.index,
        )

    w = (
        gross_target
        * score
        / score.abs().sum()
    )

    # Iteratively cap while preserving sign.
    for _ in range(10):
        over = (
            w.abs()
            > max_abs_weight
        )

        if not over.any():
            break

        capped = (
            w.where(
                ~over,
                np.sign(w)
                * max_abs_weight,
            )
        )

        used = (
            capped[
                over
            ].abs().sum()
        )

        free = (
            ~over
        )

        residual = max(
            gross_target
            - used,
            0.0,
        )

        if (
            free.sum() == 0
            or residual <= 0
        ):
            w = capped
            break

        free_scores = (
            score[
                free
            ]
        )

        if (
            free_scores.abs().sum()
            <= 0
        ):
            w = capped
            break

        capped.loc[
            free
        ] = (
            residual
            * free_scores
            / free_scores.abs().sum()
        )

        w = capped

    # Final safety rescale.
    if (
        w.abs().sum()
        > gross_target
        + 1e-12
    ):
        w = (
            gross_target
            * w
            / w.abs().sum()
        )

    return w


def build_factor_allocations(
    total_forecasts,
    factor_returns,
):
    allocation_rows = []
    signal_rows = []

    dates = (
        total_forecasts
        .index
        .get_level_values(
            "forecast_date"
        )
        .unique()
        .sort_values()
    )

    for dt in dates:
        f = (
            total_forecasts
            .xs(
                dt,
                level="forecast_date",
            )
            .reindex(
                FACTOR_NAMES
            )
        )

        signal = (
            signed_factor_signal(
                f,
                confidence_threshold=
                    MEAN_CONFIDENCE_THRESHOLD,
                expected_return_hurdle=
                    EXPECTED_RETURN_HURDLE,
            )
        )

        vol = (
            trailing_factor_vol(
                factor_returns,
                dt,
                lookback=
                    FACTOR_VOL_LOOKBACK,
                min_obs=
                    FACTOR_VOL_MIN_OBS,
            )
        )

        raw_score = pd.Series(
            0.0,
            index=FACTOR_NAMES,
        )

        valid = (
            vol.notna()
            & (vol > 0)
        )

        raw_score.loc[
            valid
        ] = (
            signal.loc[
                valid,
                "total_mean",
            ]
            / vol.loc[
                valid
            ]
        )

        raw_score = (
            raw_score
            * signal[
                "side"
            ].abs()
        )

        allocation = (
            capped_signed_normalize(
                raw_score,
                max_abs_weight=
                    MAX_FACTOR_ALLOCATION,
                gross_target=
                    TOTAL_FACTOR_GROSS,
            )
        )

        allocation_rows.append(
            pd.Series(
                allocation,
                name=dt,
            )
        )

        tmp_signal = signal.copy()

        tmp_signal[
            "factor_vol"
        ] = vol

        tmp_signal[
            "raw_score"
        ] = raw_score

        tmp_signal[
            "allocation"
        ] = allocation

        tmp_signal[
            "forecast_date"
        ] = dt

        tmp_signal[
            "factor"
        ] = tmp_signal.index

        signal_rows.append(
            tmp_signal.reset_index(
                drop=True
            )
        )

    allocations = (
        pd.DataFrame(
            allocation_rows
        )
        .sort_index()
    )

    signal_detail = (
        pd.concat(
            signal_rows,
            ignore_index=True,
        )
        .set_index(
            [
                "forecast_date",
                "factor",
            ]
        )
        .sort_index()
    )

    return allocations, signal_detail


cond_factor_alloc, cond_signal_detail = (
    build_factor_allocations(
        conditional_total_forecasts,
        factor_total_returns,
    )
)

uncond_factor_alloc, uncond_signal_detail = (
    build_factor_allocations(
        unconditional_total_forecasts,
        factor_total_returns,
    )
)

display(
    cond_signal_detail.tail(9).round(4)
)


## 22. Translate factor allocations into maturity-overlay weights

In [ ]:
def combine_factor_and_maturity_weights(
    factor_allocations,
    mimic_weights,
    max_underlying_gross,
):
    rows = []

    for dt in factor_allocations.index:
        assets = next(
            iter(
                mimic_weights.values()
            )
        ).columns

        combined = pd.Series(
            0.0,
            index=assets,
        )

        for factor in FACTOR_NAMES:
            if (
                dt
                not in mimic_weights[
                    factor
                ].index
            ):
                continue

            w_factor = (
                mimic_weights[
                    factor
                ].loc[
                    dt,
                    assets,
                ]
                .fillna(
                    0.0
                )
            )

            combined = (
                combined
                + factor_allocations.loc[
                    dt,
                    factor,
                ]
                * w_factor
            )

        gross = float(
            combined.abs().sum()
        )

        if (
            gross
            > max_underlying_gross
            and gross > 0
        ):
            combined = (
                combined
                * max_underlying_gross
                / gross
            )

        rows.append(
            pd.Series(
                combined,
                name=dt,
            )
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_index()
    )


cond_overlay_w = (
    combine_factor_and_maturity_weights(
        cond_factor_alloc,
        mimic_weights,
        MAX_UNDERLYING_GROSS,
    )
)

uncond_overlay_w = (
    combine_factor_and_maturity_weights(
        uncond_factor_alloc,
        mimic_weights,
        MAX_UNDERLYING_GROSS,
    )
)

display(
    cond_overlay_w.tail().round(3)
)


## 23. Apply overlays out of sample

In [ ]:
def apply_overlay_next_month(
    overlay_weights,
    asset_total_returns,
    asset_excess_returns,
    cash_returns,
    cost_bps,
    core_returns=None,
    overlay_scale=1.0,
):
    strategy = {}
    overlay_pnl = {}
    turnover = {}

    previous_w = None

    for dt in overlay_weights.index:
        future_dates = (
            asset_total_returns.index[
                asset_total_returns.index > dt
            ]
        )

        if len(future_dates) == 0:
            continue

        next_dt = future_dates[0]

        w = (
            overlay_weights.loc[dt]
            * overlay_scale
        )

        if previous_w is None:
            one_way_turnover = (
                0.5 * w.abs().sum()
            )
        else:
            aligned = pd.concat(
                [
                    previous_w.rename("prev"),
                    w.rename("curr"),
                ],
                axis=1,
            ).fillna(0.0)

            one_way_turnover = (
                0.5
                * (
                    aligned["curr"]
                    - aligned["prev"]
                )
                .abs()
                .sum()
            )

        # Self-financing overlay P&L uses excess returns.
        pnl = float(
            w
            @ asset_excess_returns.loc[
                next_dt,
                w.index,
            ]
        )

        cost = (
            one_way_turnover
            * cost_bps
            / 10000.0
        )

        if core_returns is None:
            base = float(
                cash_returns.loc[next_dt]
            )
        else:
            base = float(
                core_returns.loc[next_dt]
            )

        strategy[next_dt] = (
            base + pnl - cost
        )

        overlay_pnl[next_dt] = pnl
        turnover[next_dt] = one_way_turnover

        previous_w = w

    return (
        pd.Series(
            strategy,
            name="strategy_return",
        ).sort_index(),
        pd.Series(
            overlay_pnl,
            name="overlay_pnl",
        ).sort_index(),
        pd.Series(
            turnover,
            name="turnover",
        ).sort_index(),
    )


ew_core = (
    zc_returns
    .mean(axis=1)
    .rename("EqualWeightCore")
)

cond_cash_overlay_ret, cond_cash_overlay_pnl, cond_turnover = (
    apply_overlay_next_month(
        overlay_weights=cond_overlay_w,
        asset_total_returns=zc_returns,
        asset_excess_returns=zc_excess,
        cash_returns=cash_return,
        cost_bps=BASE_COST_BPS,
        core_returns=None,
        overlay_scale=1.0,
    )
)

uncond_cash_overlay_ret, _, uncond_turnover = (
    apply_overlay_next_month(
        overlay_weights=uncond_overlay_w,
        asset_total_returns=zc_returns,
        asset_excess_returns=zc_excess,
        cash_returns=cash_return,
        cost_bps=BASE_COST_BPS,
        core_returns=None,
        overlay_scale=1.0,
    )
)

cond_core_overlay_ret, cond_core_overlay_pnl, cond_core_turnover = (
    apply_overlay_next_month(
        overlay_weights=cond_overlay_w,
        asset_total_returns=zc_returns,
        asset_excess_returns=zc_excess,
        cash_returns=cash_return,
        cost_bps=BASE_COST_BPS,
        core_returns=ew_core,
        overlay_scale=CORE_OVERLAY_SCALE,
    )
)


## 24. Carry-only factor baseline

In [ ]:
def build_carry_only_forecasts(
    factor_carry_forward,
    reference_curve_sigma,
):
    rows = []

    dates = (
        factor_carry_forward.index
        .intersection(
            reference_curve_sigma.index
        )
    )

    for dt in dates:
        carry = (
            factor_carry_forward.loc[
                dt,
                FACTOR_NAMES,
            ]
        )

        # Carry is observed at the forecast date.
        # A tiny numerical SE makes its sign effectively known.
        mean_se = pd.Series(
            1e-12,
            index=FACTOR_NAMES,
        )

        out = pd.DataFrame(
            {
                "carry": carry,
                "curve_mean": 0.0,
                "curve_mean_se": mean_se,
                "curve_sigma":
                    reference_curve_sigma.loc[
                        dt,
                        FACTOR_NAMES,
                    ],
            }
        )

        out[
            "total_mean"
        ] = out[
            "carry"
        ]

        out[
            "forecast_date"
        ] = dt

        out[
            "factor"
        ] = out.index

        rows.append(
            out.reset_index(
                drop=True
            )
        )

    return (
        pd.concat(
            rows,
            ignore_index=True,
        )
        .set_index(
            [
                "forecast_date",
                "factor",
            ]
        )
        .sort_index()
    )


conditional_sigma_wide = (
    conditional_curve_forecasts[
        "curve_sigma"
    ]
    .unstack(
        "factor"
    )
)

carry_only_forecasts = (
    build_carry_only_forecasts(
        factor_carry_forward,
        conditional_sigma_wide,
    )
)

carry_factor_alloc, carry_signal_detail = (
    build_factor_allocations(
        carry_only_forecasts,
        factor_total_returns,
    )
)

carry_overlay_w = (
    combine_factor_and_maturity_weights(
        carry_factor_alloc,
        mimic_weights,
        MAX_UNDERLYING_GROSS,
    )
)

carry_cash_overlay_ret, _, carry_turnover = (
    apply_overlay_next_month(
        overlay_weights=
            carry_overlay_w,
        asset_total_returns=
            zc_returns,
        asset_excess_returns=
            zc_excess,
        cash_returns=
            cash_return,
        cost_bps=
            BASE_COST_BPS,
        core_returns=None,
        overlay_scale=1.0,
    )
)


## 25. Performance analytics

In [ ]:
def max_drawdown(
    returns,
):
    wealth = (
        1.0
        + returns
    ).cumprod()

    peak = wealth.cummax()

    dd = (
        wealth
        / peak
        - 1.0
    )

    return float(
        dd.min()
    )


def performance_metrics(
    returns,
    cash=None,
    turnover=None,
):
    r = (
        returns
        .dropna()
    )

    if len(r) == 0:
        return pd.Series(
            dtype=float
        )

    years = (
        len(r)
        / 12.0
    )

    cagr = (
        (
            1.0
            + r
        ).prod()
        ** (
            1.0
            / years
        )
        - 1.0
    )

    ann_vol = (
        r.std(
            ddof=1
        )
        * np.sqrt(
            12.0
        )
    )

    if cash is not None:
        ex = (
            r
            - cash.reindex(
                r.index
            ).fillna(
                0.0
            )
        )
    else:
        ex = r

    ex_vol = ex.std(
        ddof=1
    )

    sharpe = (
        np.sqrt(
            12.0
        )
        * ex.mean()
        / ex_vol
        if ex_vol > 0
        else np.nan
    )

    downside = ex[
        ex < 0
    ]

    sortino = (
        np.sqrt(
            12.0
        )
        * ex.mean()
        / downside.std(
            ddof=1
        )
        if (
            len(
                downside
            ) > 1
            and downside.std(
                ddof=1
            ) > 0
        )
        else np.nan
    )

    avg_turnover = (
        float(
            turnover.reindex(
                r.index
            ).mean()
        )
        if turnover is not None
        else np.nan
    )

    return pd.Series(
        {
            "CAGR":
                cagr,
            "AnnVol":
                ann_vol,
            "Sharpe":
                sharpe,
            "Sortino":
                sortino,
            "MaxDD":
                max_drawdown(
                    r
                ),
            "AvgMonthlyTurnover":
                avg_turnover,
        }
    )


oos_index = (
    cond_cash_overlay_ret.index
)

cash_bench = (
    cash_return.reindex(
        oos_index
    )
)

ew_bench = (
    ew_core.reindex(
        oos_index
    )
)

zc10_bench = (
    zc_returns[
        "ZC10Y"
    ].reindex(
        oos_index
    )
)

results = pd.DataFrame(
    {
        "Conditional_CashOverlay":
            performance_metrics(
                cond_cash_overlay_ret,
                cash=
                    cash_bench,
                turnover=
                    cond_turnover,
            ),
        "Unconditional_CashOverlay":
            performance_metrics(
                uncond_cash_overlay_ret.reindex(
                    oos_index
                ),
                cash=
                    cash_bench,
                turnover=
                    uncond_turnover,
            ),
        "CarryOnly_CashOverlay":
            performance_metrics(
                carry_cash_overlay_ret.reindex(
                    oos_index
                ),
                cash=
                    cash_bench,
                turnover=
                    carry_turnover,
            ),
        "Conditional_CorePlusOverlay":
            performance_metrics(
                cond_core_overlay_ret.reindex(
                    oos_index
                ),
                cash=
                    cash_bench,
                turnover=
                    cond_core_turnover,
            ),
        "EqualWeightCore":
            performance_metrics(
                ew_bench,
                cash=
                    cash_bench,
            ),
        "ZC10Y":
            performance_metrics(
                zc10_bench,
                cash=
                    cash_bench,
            ),
        "Cash":
            performance_metrics(
                cash_bench,
                cash=
                    cash_bench,
            ),
    }
).T

display(
    results.round(4)
)


## 26. Equity curves

In [ ]:
curves = pd.DataFrame(
    {
        "Conditional Cash Overlay":
            (
                1.0
                + cond_cash_overlay_ret
            ).cumprod(),
        "Unconditional Cash Overlay":
            (
                1.0
                + uncond_cash_overlay_ret.reindex(
                    oos_index
                )
            ).cumprod(),
        "Carry-only Cash Overlay":
            (
                1.0
                + carry_cash_overlay_ret.reindex(
                    oos_index
                )
            ).cumprod(),
        "Core + Conditional Overlay":
            (
                1.0
                + cond_core_overlay_ret.reindex(
                    oos_index
                )
            ).cumprod(),
        "Equal-weight Core":
            (
                1.0
                + ew_bench
            ).cumprod(),
        "Cash":
            (
                1.0
                + cash_bench
            ).cumprod(),
    }
)

ax = curves.plot(
    figsize=(12, 6)
)

ax.set_title(
    "v0.4 real-data OOS cumulative growth"
)

ax.set_ylabel(
    "Growth of $1"
)

plt.show()


## 27. Conditional value-add tests

In [ ]:
common_idx = (
    cond_cash_overlay_ret.index
    .intersection(
        uncond_cash_overlay_ret.index
    )
)

cond_minus_uncond = (
    cond_cash_overlay_ret.loc[
        common_idx
    ]
    - uncond_cash_overlay_ret.loc[
        common_idx
    ]
)

cond_minus_carry = (
    cond_cash_overlay_ret.loc[
        common_idx
    ]
    - carry_cash_overlay_ret.reindex(
        common_idx
    )
)

def mean_t_stat(
    x: pd.Series,
):
    x = x.dropna()

    return float(
        x.mean()
        / (
            x.std(
                ddof=1
            )
            / np.sqrt(
                len(x)
            )
        )
    )


print(
    "Conditional - Unconditional annualized:",
    round(
        float(
            cond_minus_uncond.mean()
            * 12.0
        ),
        4,
    ),
)

print(
    "Conditional - Unconditional t-stat:",
    round(
        mean_t_stat(
            cond_minus_uncond
        ),
        3,
    ),
)

print(
    "Conditional - Carry-only annualized:",
    round(
        float(
            cond_minus_carry.mean()
            * 12.0
        ),
        4,
    ),
)

print(
    "Conditional - Carry-only t-stat:",
    round(
        mean_t_stat(
            cond_minus_carry
        ),
        3,
    ),
)


## 28. HAC alpha of core + overlay versus core

In [ ]:
reg_df = pd.concat(
    [
        cond_core_overlay_ret.rename(
            "strategy"
        ),
        ew_core.rename(
            "core"
        ),
        cash_return.rename(
            "cash"
        ),
    ],
    axis=1,
).dropna()

reg_df[
    "strategy_excess"
] = (
    reg_df[
        "strategy"
    ]
    - reg_df[
        "cash"
    ]
)

reg_df[
    "core_excess"
] = (
    reg_df[
        "core"
    ]
    - reg_df[
        "cash"
    ]
)

X = sm.add_constant(
    reg_df[
        "core_excess"
    ]
)

ols = sm.OLS(
    reg_df[
        "strategy_excess"
    ],
    X,
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3
    },
)

print(
    "Core+overlay annualized HAC alpha:",
    round(
        float(
            ols.params[
                "const"
            ]
            * 12.0
        ),
        4,
    ),
)

print(
    "HAC alpha t-stat:",
    round(
        float(
            ols.tvalues[
                "const"
            ]
        ),
        3,
    ),
)

print(
    "Core beta:",
    round(
        float(
            ols.params[
                "core_excess"
            ]
        ),
        3,
    ),
)


## 29. Transaction-cost sensitivity

In [ ]:
cost_rows = []

for cost_bps in COST_SENSITIVITY_BPS:

    r, _, to = (
        apply_overlay_next_month(
            overlay_weights=
                cond_overlay_w,
            asset_total_returns=
                zc_returns,
            asset_excess_returns=
                zc_excess,
            cash_returns=
                cash_return,
            cost_bps=
                cost_bps,
            core_returns=None,
            overlay_scale=1.0,
        )
    )

    m = performance_metrics(
        r,
        cash=
            cash_return.reindex(
                r.index
            ),
        turnover=
            to,
    )

    m.name = (
        f"{cost_bps:.0f} bps"
    )

    cost_rows.append(
        m
    )

cost_table = pd.DataFrame(
    cost_rows
)

display(
    cost_table.round(4)
)


## 30. Subperiod robustness

In [ ]:
def subperiod_metrics(
    strategy,
    benchmark,
    cash,
):
    periods = {
        "Full": (
            strategy.index.min(),
            strategy.index.max(),
        ),
        "Pre-2020": (
            strategy.index.min(),
            pd.Timestamp(
                "2019-12-31"
            ),
        ),
        "2020+": (
            pd.Timestamp(
                "2020-01-31"
            ),
            strategy.index.max(),
        ),
    }

    rows = []

    for name, (
        start,
        end,
    ) in periods.items():

        s = strategy.loc[
            (
                strategy.index
                >= start
            )
            & (
                strategy.index
                <= end
            )
        ]

        b = benchmark.reindex(
            s.index
        )

        c = cash.reindex(
            s.index
        )

        smet = performance_metrics(
            s,
            cash=c,
        )

        bmet = performance_metrics(
            b,
            cash=c,
        )

        rows.append(
            pd.Series(
                {
                    "Strategy_CAGR":
                        smet[
                            "CAGR"
                        ],
                    "Strategy_Sharpe":
                        smet[
                            "Sharpe"
                        ],
                    "Strategy_MaxDD":
                        smet[
                            "MaxDD"
                        ],
                    "Benchmark_CAGR":
                        bmet[
                            "CAGR"
                        ],
                    "Benchmark_Sharpe":
                        bmet[
                            "Sharpe"
                        ],
                    "Benchmark_MaxDD":
                        bmet[
                            "MaxDD"
                        ],
                },
                name=name,
            )
        )

    return pd.DataFrame(
        rows
    )


display(
    subperiod_metrics(
        cond_core_overlay_ret,
        ew_core,
        cash_return,
    ).round(4)
)


## 31. Factor signal attribution

In [ ]:
signal_attribution = (
    cond_signal_detail
    .reset_index()
    .groupby(
        "factor"
    )
    .agg(
        avg_total_mean=(
            "total_mean",
            "mean",
        ),
        avg_abs_total_mean=(
            "total_mean",
            lambda x:
                x.abs().mean(),
        ),
        avg_factor_vol=(
            "factor_vol",
            "mean",
        ),
        avg_abs_allocation=(
            "allocation",
            lambda x:
                x.abs().mean(),
        ),
        active_fraction=(
            "side",
            lambda x:
                (
                    x != 0
                ).mean(),
        ),
    )
)

display(
    signal_attribution.round(4)
)


## 31A. Mean-confidence diagnostics

This table distinguishes confidence about the **expected premium** from the probability that the next realized return has the expected sign.

A persistent risk premium may have high mean confidence while `p_next_positive_return` remains close to 50%.


In [ ]:
mean_confidence_summary = (
    cond_signal_detail
    .reset_index()
    .groupby("factor")
    .agg(
        avg_abs_mean=(
            "total_mean",
            lambda x: x.abs().mean(),
        ),
        avg_mean_se=(
            "mean_se",
            "mean",
        ),
        avg_p_next_positive=(
            "p_next_positive_return",
            "mean",
        ),
        active_fraction=(
            "side",
            lambda x: (x != 0).mean(),
        ),
    )
)

display(
    mean_confidence_summary.round(4)
)


## 32. Carry versus conditional curve component

In [ ]:
carry_curve_summary = pd.DataFrame(
    {
        "avg_carry":
            conditional_total_forecasts[
                "carry"
            ]
            .groupby(
                level="factor"
            )
            .mean(),
        "avg_abs_carry":
            conditional_total_forecasts[
                "carry"
            ]
            .abs()
            .groupby(
                level="factor"
            )
            .mean(),
        "avg_curve_forecast":
            conditional_total_forecasts[
                "curve_mean"
            ]
            .groupby(
                level="factor"
            )
            .mean(),
        "avg_abs_curve_forecast":
            conditional_total_forecasts[
                "curve_mean"
            ]
            .abs()
            .groupby(
                level="factor"
            )
            .mean(),
    }
)

display(
    carry_curve_summary.round(5)
)


## 33. Latest factor state and portfolio

In [ ]:
latest_dt = (
    cond_factor_alloc.index[-1]
)

print(
    "Latest forecast date:",
    latest_dt.date(),
)

print(
    "\nFactor allocations"
)

display(
    cond_factor_alloc.loc[
        latest_dt
    ]
    .to_frame(
        "allocation"
    )
    .round(4)
)

print(
    "\nUnderlying maturity overlay"
)

display(
    cond_overlay_w.loc[
        latest_dt
    ]
    .to_frame(
        "weight"
    )
    .round(4)
)

print(
    "\nSignal detail"
)

display(
    cond_signal_detail.xs(
        latest_dt,
        level="forecast_date",
    ).round(4)
)


## 34. ETF scaling fix from v0.3

The v0.3 ETF robustness test regressed ETF returns on raw PCA scores, while the zero-coupon factor betas were defined relative to **one-sigma PCA shocks**.

The correctly scaled market factors are therefore:

\[
\widetilde f_{k,t}
=
\frac{
f_{k,t}
}{
\sqrt{\lambda_k}
}.
\]

`factor_scores_1sigma` in this notebook is the corrected factor series to use in any future ETF beta regression.

The main v0.5 strategy does not rely on the flawed v0.3 ETF transfer. ETF implementation will be reintroduced only after the factor-mimicking premium signal itself is validated.


In [ ]:
print(
    "Standard deviation of corrected one-sigma PCA factors:"
)

display(
    factor_scores_1sigma.std(
        ddof=1
    ).to_frame(
        "std"
    ).round(3)
)


## 35. Integration with the Fixed-Income Pricing & Risk Engine

The current standalone notebook uses zero-coupon analytics because they make the empirical test transparent.

For production-grade instruments, the existing Fixed-Income Pricing & Risk Engine can replace the analytic approximations:

### Existing reusable components

- `key_rate_risk()`  
  Key-rate DV01 and diagonal convexity.

- `key_rate_convexity_matrix()`  
  Full key-rate gamma structure including cross terms.

- `curve_factor_risk()`  
  Accepts externally supplied PCA factor loadings.

- `carry_roll_down()`  
  Deterministic carry/roll using the current term structure.

- `explain_curve_pnl()`  
  Full repricing versus first- and second-order curve P&L.

- `bucketed_surface_vega()`  
  Separate implied-volatility risk layer.

Therefore the intended production architecture is:

\[
\boxed{
\text{Empirical rates research}
}
\]

\[
\text{state}
\rightarrow
\text{factor premia}
\rightarrow
\text{signals}
\rightarrow
\text{target factor exposures}
\]

feeding:

\[
\boxed{
\text{Pricing \& risk engine}
}
\]

\[
\text{instrument selection}
\rightarrow
KR01/DV01
\rightarrow
convexity
\rightarrow
Vega
\rightarrow
full repricing
\rightarrow
P\&L explain.
\]


## 36. Verdict block

In [ ]:
def verdict_table(
    results,
    cost_table,
    cond_minus_uncond,
    cond_minus_carry,
    mimic_purity,
):
    checks = {}

    checks[
        "Exact cross-factor neutrality"
    ] = (
        mimic_purity[
            "cross_factor_loading_norm"
        ].max()
        < 1e-8
    )

    checks[
        "Factor purity median > 0.999"
    ] = (
        mimic_purity[
            "purity_normalized_beta"
        ].median()
        > 0.999
    )

    checks[
        "Funding identity exact"
    ] = (
        (
            mimic_purity[
                "risky_net_weight"
            ]
            + mimic_purity[
                "cash_funding_weight"
            ]
        )
        .abs()
        .max()
        < 1e-12
    )

    checks[
        "Conditional overlay Sharpe > 0"
    ] = (
        results.loc[
            "Conditional_CashOverlay",
            "Sharpe",
        ] > 0
    )

    checks[
        "Conditional > unconditional mean"
    ] = (
        cond_minus_uncond.mean() > 0
    )

    checks[
        "Conditional > carry-only mean"
    ] = (
        cond_minus_carry.mean() > 0
    )

    checks[
        "Core+overlay Sharpe > core"
    ] = (
        results.loc[
            "Conditional_CorePlusOverlay",
            "Sharpe",
        ]
        >
        results.loc[
            "EqualWeightCore",
            "Sharpe",
        ]
    )

    checks[
        "Positive Sharpe at 10 bps"
    ] = (
        cost_table.loc[
            "10 bps",
            "Sharpe",
        ] > 0
    )

    return pd.DataFrame(
        {"Pass": pd.Series(checks)}
    )


display(
    verdict_table(
        results,
        cost_table,
        cond_minus_uncond,
        cond_minus_carry,
        mimic_purity,
    )
)


## 37. Research interpretation before changing parameters

v0.7 tests a materially different hypothesis from v0.6.

### A. Persistent risk premia

A one-month factor can have an economically meaningful long-run premium while the next realized monthly return remains almost unpredictable in sign.

Therefore failure of:

\[
P(R_{t+1}>0)>60\%
\]

does **not** imply absence of a risk premium.

### B. Mean confidence

The new gate asks whether the estimated conditional mean itself is credibly positive or negative.

If the unconditional funded Level premium is persistent, this architecture should be able to retain it rather than suppress it because monthly return volatility is high.

### C. Conditional timing value

The relevant comparison is now:

- unconditional mean-confidence strategy;
- carry-only;
- conditional mean-confidence strategy.

If conditional still fails to beat unconditional, that is stronger evidence that the current state variables do not add timing information.

### D. What comes next if timing still fails

Do not optimize the confidence threshold.

Move to empirically motivated bond-return predictors:

- forward-rate / forward-spread signals;
- Fama–Bliss style predictors;
- Cochrane–Piazzesi style common return predictor;
- explicit term-premium measures;
- longer forecast horizons;
- regime-dependent expected returns.

That would be a change in the **asset-pricing signal model**, not cosmetic portfolio tuning.


The v0.5 run should be interpreted layer by layer.

### A. Factor construction

If factor purity is poor, the maturity cross-section still does not span Level/Slope/Curvature cleanly enough. Do not tune the forecast model.

### B. Carry/roll economics

If carry-only works but conditional curve forecasts do not, the profitable component is structural term-structure carry rather than dynamic factor timing.

### C. Conditional timing

If conditional beats both unconditional and carry-only, then the state layer is adding genuine predictive information.

### D. Portfolio construction

If factor signals look predictive but the overlay loses after costs, the research problem has moved from signal discovery to turnover / implementation / instrument selection.

### E. Core-plus-overlay

If the pure factor overlay is modest but improves the equal-weight maturity core, the economically relevant application may be **systematic portfolio construction** rather than standalone alpha.

No hyperparameter search should be performed until these distinctions are clear.


## 38. Saved outputs

In [ ]:
OUTPUT_DIR = Path(
    "./rates_factor_timing_outputs_v07"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)

results.to_csv(
    OUTPUT_DIR
    / "performance.csv"
)

cost_table.to_csv(
    OUTPUT_DIR
    / "cost_sensitivity.csv"
)

factor_total_returns.to_csv(
    OUTPUT_DIR
    / "factor_total_returns.csv"
)

factor_carry_forward.to_csv(
    OUTPUT_DIR
    / "factor_carry_forward.csv"
)

factor_curve_realized.to_csv(
    OUTPUT_DIR
    / "factor_curve_realized.csv"
)

mimic_purity.to_csv(
    OUTPUT_DIR
    / "factor_mimicking_purity.csv"
)

cond_factor_alloc.to_csv(
    OUTPUT_DIR
    / "conditional_factor_allocations.csv"
)

cond_overlay_w.to_csv(
    OUTPUT_DIR
    / "conditional_maturity_overlay_weights.csv"
)

conditional_total_forecasts.to_csv(
    OUTPUT_DIR
    / "conditional_factor_forecasts.csv"
)

cond_signal_detail.to_csv(
    OUTPUT_DIR
    / "conditional_signal_detail.csv"
)

print(
    "Saved outputs to:",
    OUTPUT_DIR.resolve(),
)


## 39. Next extensions after the v0.7 mean-confidence test

The natural next extensions are not arbitrary extra predictors. They are economically distinct premium layers:

1. **Explicit term premium**  
   Separate expected short-rate path from term compensation.

2. **Inflation risk premium**  
   Market-implied inflation versus physical inflation expectation.

3. **Rate-volatility premium**  
   \(Q-P\) variance / volatility compensation from option-implied versus physical rate volatility.

4. **Regime-mixture conditional distributions**  
   Replace one Gaussian conditional model with soft regime probabilities.

5. **Exact instrument implementation**  
   Use the Fixed-Income Pricing & Risk Engine for coupon bonds, swaps, futures, caps/floors and swaptions.

6. **Cross-asset extension**  
   Reuse the common conditional-risk-premium interface for equity, credit, FX and commodities.

The architecture remains:

\[
\boxed{
\text{research signal}
\rightarrow
\text{priced risk}
\rightarrow
\text{expected compensation}
\rightarrow
\text{portfolio exposure}
\rightarrow
\text{full repricing / realized P\&L}
}
\]


# v0.8 — Empirical Bond-Premia Research Layer

The sections below leave the funded factor portfolios, risk scaling, transaction-cost accounting and monthly implementation unchanged.

Only the expected-return model is replaced and tested across 3-, 6- and 12-month horizons.


## 40. Forward rates and Fama–Bliss spreads

For continuously compounded zero yields, the one-year forward rate between years \(n-1\) and \(n\) is:

\[
f_t^{(n)}
=
ny_t^{(n)}-(n-1)y_t^{(n-1)}.
\]

The Fama–Bliss forward spread is:

\[
FB_t^{(n)}
=
f_t^{(n)}-y_t^{(1)}.
\]

The standard 2Y–5Y forward-spread block is used as a low-dimensional predictor family. These features are observable at the forecast date and do not require future data.


In [ ]:
def build_forward_rate_panel(
    zero_curve: pd.DataFrame,
    max_maturity: int = 10,
):
    out = pd.DataFrame(index=zero_curve.index)

    out["f1"] = zero_curve["1Y"]

    for n in range(2, max_maturity + 1):
        out[f"f{n}"] = (
            n * zero_curve[f"{n}Y"]
            - (n - 1) * zero_curve[f"{n-1}Y"]
        )

    return out


forward_rates = build_forward_rate_panel(
    zero_monthly,
    max_maturity=10,
)

fb_features = pd.DataFrame(
    index=forward_rates.index
)

for n in FB_FORWARD_MATURITIES:
    fb_features[f"FB{n}"] = (
        forward_rates[f"f{n}"]
        - forward_rates["f1"]
    )

cp_forward_features = forward_rates[
    [f"f{n}" for n in CP_FORWARD_MATURITIES]
].copy()

print("Latest forward rates")
display(forward_rates.tail().round(5))

print("Latest Fama-Bliss spreads")
display(fb_features.tail().round(5))


## 41. Direct multi-horizon zero-coupon holding returns

For horizon \(H\) months, the bond bought at date \(t\) with maturity \(\tau\) is repriced at \(t+H\) with remaining maturity:

\[
\tau - H/12.
\]

The realized bond return is computed directly from prices rather than compounding one-month returns.

Two funding objects are kept separate:

- **realized cash return** over the future \(H\) months, used to calculate realized excess returns;
- **known funding carry** formed from the current policy-rate level and compounded over \(H\) months, used only for the ex-ante carry decomposition.

Any subsequent change in short-term funding rates therefore enters the realized residual rather than leaking into ex-ante carry.


In [ ]:
current_funding_monthly = (
    state_raw["FEDFUNDS"]
    .resample("ME")
    .last()
    .reindex(zero_monthly.index)
    .ffill()
    / 100.0
    / 12.0
)


def build_horizon_zero_coupon_data(
    zero_curve: pd.DataFrame,
    maturities: list[int],
    cash_return_monthly: pd.Series,
    current_funding_monthly: pd.Series,
    horizon_months: int,
):
    h_years = horizon_months / 12.0

    realized_rows = []
    excess_rows = []
    log_excess_rows = []
    carry_rows = []
    realized_cash_rows = []
    known_cash_rows = []

    idx = zero_curve.index

    for i in range(len(idx) - horizon_months):
        dt0 = idx[i]
        dt1 = idx[i + horizon_months]

        curve0 = zero_curve.loc[dt0]
        curve1 = zero_curve.loc[dt1]

        cash_dates = idx[i + 1 : i + horizon_months + 1]
        cash_path = cash_return_monthly.reindex(cash_dates)

        if cash_path.isna().any():
            continue

        realized_cash = float(
            (1.0 + cash_path).prod() - 1.0
        )

        funding_m = current_funding_monthly.loc[dt0]

        if pd.isna(funding_m):
            continue

        known_cash = float(
            (1.0 + funding_m) ** horizon_months - 1.0
        )

        realized = {}
        excess = {}
        log_excess = {}
        funded_carry = {}

        for tau in maturities:
            remaining = tau - h_years

            if remaining < 1.0:
                continue

            asset = f"ZC{tau}Y"

            y0_tau = float(
                curve0[f"{tau}Y"]
            )

            y0_remaining = interp_curve_yield(
                curve0,
                remaining,
            )

            y1_remaining = interp_curve_yield(
                curve1,
                remaining,
            )

            p0 = zero_coupon_price_cc(
                y0_tau,
                tau,
            )

            p1 = zero_coupon_price_cc(
                y1_remaining,
                remaining,
            )

            p_carry = zero_coupon_price_cc(
                y0_remaining,
                remaining,
            )

            r_total = p1 / p0 - 1.0
            r_bond_carry = p_carry / p0 - 1.0

            realized[asset] = r_total
            excess[asset] = r_total - realized_cash

            # Academic bond-return regressions are conventionally written
            # in log excess returns. Strategy P&L remains in simple returns.
            log_excess[asset] = (
                np.log1p(r_total)
                - np.log1p(realized_cash)
            )

            funded_carry[asset] = r_bond_carry - known_cash

        realized_rows.append(
            pd.Series(realized, name=dt0)
        )

        excess_rows.append(
            pd.Series(excess, name=dt0)
        )

        log_excess_rows.append(
            pd.Series(log_excess, name=dt0)
        )

        carry_rows.append(
            pd.Series(funded_carry, name=dt0)
        )

        realized_cash_rows.append(
            pd.Series({"cash_realized": realized_cash}, name=dt0)
        )

        known_cash_rows.append(
            pd.Series({"cash_known": known_cash}, name=dt0)
        )

    return {
        "realized": pd.DataFrame(realized_rows).sort_index(),
        "excess": pd.DataFrame(excess_rows).sort_index(),
        "log_excess": pd.DataFrame(log_excess_rows).sort_index(),
        "funded_carry": pd.DataFrame(carry_rows).sort_index(),
        "realized_cash": pd.DataFrame(realized_cash_rows)["cash_realized"].sort_index(),
        "known_cash": pd.DataFrame(known_cash_rows)["cash_known"].sort_index(),
    }


horizon_asset_data = {
    h: build_horizon_zero_coupon_data(
        zero_curve=zero_monthly,
        maturities=TRADED_MATURITIES,
        cash_return_monthly=cash_return,
        current_funding_monthly=current_funding_monthly,
        horizon_months=h,
    )
    for h in PREDICTION_HORIZONS
}

for h in PREDICTION_HORIZONS:
    print(
        f"H={h} months | forecast dates =",
        len(horizon_asset_data[h]["excess"]),
    )


## 42. Multi-horizon funded factor targets

The exact funded factor weights constructed by the stable v0.7 stack are held fixed at forecast date \(t\) and applied to the future \(H\)-month maturity excess returns:

\[
R^{factor,H}_{k,t\rightarrow t+H}
=
w_{k,t}'R^{excess,H}_{t\rightarrow t+H}.
\]

Known funded carry is:

\[
CR^{factor,H}_{k,t}
=
w_{k,t}'CR^{funded,H}_t.
\]

The regression target remains the residual curve component:

\[
Curve^{factor,H}_{k,t\rightarrow t+H}
=
R^{factor,H}_{k,t\rightarrow t+H}
-
CR^{factor,H}_{k,t}.
\]


In [ ]:
def build_horizon_factor_targets(
    mimic_weights,
    asset_data,
):
    excess_assets = asset_data["excess"]
    funded_carry_assets = asset_data["funded_carry"]

    total = pd.DataFrame(
        index=excess_assets.index,
        columns=FACTOR_NAMES,
        dtype=float,
    )

    carry = pd.DataFrame(
        index=excess_assets.index,
        columns=FACTOR_NAMES,
        dtype=float,
    )

    for dt in excess_assets.index:
        for factor in FACTOR_NAMES:
            if dt not in mimic_weights[factor].index:
                continue

            w = mimic_weights[factor].loc[dt].dropna()

            if len(w) == 0:
                continue

            assets = w.index.intersection(
                excess_assets.columns
            )

            if len(assets) == 0:
                continue

            total.loc[dt, factor] = float(
                w.loc[assets]
                @ excess_assets.loc[dt, assets]
            )

            carry.loc[dt, factor] = float(
                w.loc[assets]
                @ funded_carry_assets.loc[dt, assets]
            )

    curve = total - carry

    return {
        "total": total,
        "carry": carry,
        "curve": curve,
    }


horizon_factor_targets = {
    h: build_horizon_factor_targets(
        mimic_weights=mimic_weights,
        asset_data=horizon_asset_data[h],
    )
    for h in PREDICTION_HORIZONS
}

for h in PREDICTION_HORIZONS:
    print(f"H={h} factor target summary")
    display(
        horizon_factor_targets[h]["total"]
        .describe()
        .T[["mean", "std"]]
        .round(5)
    )


## 43. Fama–Bliss diagnostics at the canonical 12-month horizon

As a literature-aligned diagnostic, each maturity excess return is regressed on its corresponding forward spread at the 12-month horizon:

\[
rx_{t,t+12}^{(n)}
=
\alpha_n
+
\beta_n FB_t^{(n)}
+
\varepsilon_{t+12}^{(n)}.
\]

Because 12-month returns overlap under monthly sampling, HAC standard errors use 11 lags.


In [ ]:
def fama_bliss_univariate_diagnostics(
    asset_excess_h: pd.DataFrame,
    fb_panel: pd.DataFrame,
    maturities=(2, 3, 4, 5),
    hac_lags=11,
):
    rows = []

    for n in maturities:
        asset = f"ZC{n}Y"
        pred = f"FB{n}"

        frame = pd.concat(
            [
                asset_excess_h[asset].rename("rx"),
                fb_panel[pred].rename("fb"),
            ],
            axis=1,
        ).dropna()

        X = sm.add_constant(frame["fb"])

        fit = sm.OLS(
            frame["rx"],
            X,
        ).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags},
        )

        rows.append(
            pd.Series(
                {
                    "beta": fit.params["fb"],
                    "t_HAC": fit.tvalues["fb"],
                    "R2": fit.rsquared,
                    "n": len(frame),
                },
                name=asset,
            )
        )

    return pd.DataFrame(rows)


fb_12m_diagnostics = fama_bliss_univariate_diagnostics(
    asset_excess_h=horizon_asset_data[12]["log_excess"],
    fb_panel=fb_features,
)

display(
    fb_12m_diagnostics.round(4)
)


## 44. Leakage-safe standardized ridge predictor

The predictor model standardizes features **inside each walk-forward training sample**. This is necessary because the v0.7 state variables are z-scored while forward spreads are raw rate quantities.

The ridge covariance approximation is the same mean-confidence machinery introduced in v0.7.


In [ ]:
def fit_standardized_ridge_predictor(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    x_current: pd.Series,
    ridge: float,
    min_obs: int = 60,
):
    frame = pd.concat(
        [y_train.rename("y"), X_train],
        axis=1,
    ).dropna()

    if len(frame) < max(min_obs, X_train.shape[1] + 5):
        return None

    features = list(X_train.columns)

    y = frame["y"].to_numpy(dtype=float)
    X_raw = frame[features].astype(float)

    mu_x = X_raw.mean(axis=0)
    sd_x = X_raw.std(axis=0, ddof=1).replace(0.0, 1.0)

    Xz = X_raw.subtract(mu_x, axis=1).divide(sd_x, axis=1)

    x0 = (
        x_current.reindex(features)
        .astype(float)
        .subtract(mu_x)
        .divide(sd_x)
    )

    if x0.isna().any():
        return None

    X = np.column_stack(
        [np.ones(len(Xz)), Xz.to_numpy(dtype=float)]
    )

    x = np.r_[1.0, x0.to_numpy(dtype=float)]

    XtX = X.T @ X

    penalty = np.eye(XtX.shape[0]) * ridge
    penalty[0, 0] = 0.0

    A = XtX + penalty
    A_inv = np.linalg.pinv(A)

    beta = A_inv @ X.T @ y
    resid = y - X @ beta

    dof = max(len(y) - len(beta), 1)
    s2 = float(resid @ resid) / dof

    Vb = s2 * A_inv @ XtX @ A_inv

    pred_mean = float(x @ beta)
    pred_mean_se = sqrt(max(float(x @ Vb @ x), 1e-18))
    resid_sigma = sqrt(max(s2, 1e-18))

    return {
        "mean": pred_mean,
        "mean_se": pred_mean_se,
        "sigma": resid_sigma,
        "n": len(y),
    }


## 45. Walk-forward Cochrane–Piazzesi-style common predictor

For each forecast date and horizon, the first-stage regression is estimated using only returns already fully realized by that date.

The target is the average **log excess return** across 2Y–5Y zero-coupon bonds:

\[
\overline{rx}^{H}_t
=
\frac14\sum_{n=2}^{5}rx_t^{(n,H)}.
\]

The predictor is the fitted value from the 1Y–5Y forward curve:

\[
CP_t^H
=
\hat a_H
+
\hat\gamma_H'f_t.
\]

This is a **CP-style** implementation rather than a claim of exact replication at every non-annual horizon. The canonical paper uses annual excess returns; 3M and 6M are deliberate horizon extensions.


In [ ]:
def fit_cp_first_stage(
    train_dates,
    avg_excess_target: pd.Series,
    forward_panel: pd.DataFrame,
    current_date,
    ridge: float = 1e-10,
):
    features = list(forward_panel.columns)

    frame = pd.concat(
        [
            avg_excess_target.reindex(train_dates).rename("y"),
            forward_panel.reindex(train_dates),
        ],
        axis=1,
    ).dropna()

    if len(frame) < max(24, len(features) + 5):
        return None

    y = frame["y"].to_numpy(dtype=float)
    Xf = frame[features].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(Xf)), Xf])

    penalty = np.eye(X.shape[1]) * ridge
    penalty[0, 0] = 0.0

    beta = (
        np.linalg.pinv(X.T @ X + penalty)
        @ X.T
        @ y
    )

    cp_train = pd.Series(
        X @ beta,
        index=frame.index,
        name="CP",
    )

    x_current_raw = forward_panel.loc[
        current_date,
        features,
    ].to_numpy(dtype=float)

    x_current = np.r_[1.0, x_current_raw]

    cp_current = float(x_current @ beta)

    coefficients = pd.Series(
        beta,
        index=["intercept", *features],
    )

    return {
        "cp_train": cp_train,
        "cp_current": cp_current,
        "coefficients": coefficients,
    }


## 46. Prespecified multi-horizon model families

At forecast date \(t\), an \(H\)-month target observed at date \(s\) can enter the training sample only if:

\[
s+H \le t.
\]

This explicit realization cutoff prevents overlapping-horizon lookahead.

For every horizon and model family, the model forecasts the **future curve-repricing factor return**. Known funded carry is added afterward.


In [ ]:
def realization_cutoff_date(
    forecast_date,
    horizon_months,
    master_index,
):
    if forecast_date not in master_index:
        return None

    pos = master_index.get_loc(forecast_date)

    if isinstance(pos, slice) or pos < horizon_months:
        return None

    return master_index[pos - horizon_months]


def model_predictor_training_block(
    model_family,
    train_dates,
    current_date,
    state_panel,
    fb_panel,
    cp_payload=None,
):
    if model_family == "BASE_STATE":
        return (
            state_panel.reindex(train_dates),
            state_panel.loc[current_date],
        )

    if model_family == "FB":
        return (
            fb_panel.reindex(train_dates),
            fb_panel.loc[current_date],
        )

    if model_family == "FB_STATE":
        train = pd.concat(
            [
                fb_panel.reindex(train_dates),
                state_panel.reindex(train_dates),
            ],
            axis=1,
        )

        current = pd.concat(
            [
                fb_panel.loc[current_date],
                state_panel.loc[current_date],
            ]
        )

        return train, current

    if model_family in {"CP", "CP_STATE"}:
        if cp_payload is None:
            return None, None

        cp_train = cp_payload["cp_train"].to_frame("CP")
        cp_current = pd.Series(
            {"CP": cp_payload["cp_current"]}
        )

        if model_family == "CP":
            return cp_train, cp_current

        train = pd.concat(
            [
                cp_train,
                state_panel.reindex(cp_train.index),
            ],
            axis=1,
        )

        current = pd.concat(
            [
                cp_current,
                state_panel.loc[current_date],
            ]
        )

        return train, current

    raise ValueError(
        f"Unknown model family: {model_family}"
    )


def walk_forward_empirical_premia_model(
    horizon_months,
    model_family,
    curve_target_h,
    carry_h,
    avg_asset_excess_h,
    state_panel,
    fb_panel,
    cp_forward_panel,
    master_index,
    min_train=60,
    ridge=1.0,
):
    required_current = (
        state_panel.index
        if "STATE" in model_family
        else fb_panel.index
        if model_family in {"FB"}
        else cp_forward_panel.index
    )

    candidate_dates = (
        curve_target_h.index
        .intersection(carry_h.index)
        .intersection(required_current)
        .sort_values()
    )

    rows = []
    latest_cp_coefficients = None

    for current_date in candidate_dates:
        cutoff = realization_cutoff_date(
            current_date,
            horizon_months,
            master_index,
        )

        if cutoff is None:
            continue

        train_dates = curve_target_h.index[
            curve_target_h.index <= cutoff
        ]

        # All models need fully observed factor targets.
        train_dates = train_dates[
            curve_target_h.reindex(train_dates)
            .notna()
            .all(axis=1)
        ]

        if len(train_dates) < min_train:
            continue

        cp_payload = None

        if model_family in {"CP", "CP_STATE"}:
            cp_payload = fit_cp_first_stage(
                train_dates=train_dates,
                avg_excess_target=avg_asset_excess_h,
                forward_panel=cp_forward_panel,
                current_date=current_date,
                ridge=CP_FIRST_STAGE_RIDGE,
            )

            if cp_payload is None:
                continue

            latest_cp_coefficients = cp_payload["coefficients"]

        X_train, x_current = model_predictor_training_block(
            model_family=model_family,
            train_dates=train_dates,
            current_date=current_date,
            state_panel=state_panel,
            fb_panel=fb_panel,
            cp_payload=cp_payload,
        )

        if X_train is None or x_current is None:
            continue

        factor_rows = []

        for factor in FACTOR_NAMES:
            pred = fit_standardized_ridge_predictor(
                X_train=X_train,
                y_train=curve_target_h[factor].reindex(train_dates),
                x_current=x_current,
                ridge=ridge,
                min_obs=min_train,
            )

            if pred is None:
                continue

            factor_rows.append(
                pd.Series(
                    {
                        "curve_mean_h": pred["mean"],
                        "curve_mean_se_h": pred["mean_se"],
                        "curve_sigma_h": pred["sigma"],
                        "carry_h": carry_h.loc[current_date, factor],
                        "n_train": pred["n"],
                    },
                    name=factor,
                )
            )

        if len(factor_rows) != len(FACTOR_NAMES):
            continue

        block = pd.DataFrame(factor_rows)
        block["total_mean_h"] = (
            block["carry_h"]
            + block["curve_mean_h"]
        )

        # Convert the H-month expected-return object to a monthly-equivalent
        # signal for portfolio sizing. Sign confidence is invariant to this
        # positive scaling.
        block["total_mean"] = (
            block["total_mean_h"]
            / horizon_months
        )

        block["curve_mean"] = (
            block["curve_mean_h"]
            / horizon_months
        )

        block["curve_mean_se"] = (
            block["curve_mean_se_h"]
            / horizon_months
        )

        block["curve_sigma"] = (
            block["curve_sigma_h"]
            / np.sqrt(horizon_months)
        )

        block["forecast_date"] = current_date
        block["factor"] = block.index
        block["horizon_months"] = horizon_months
        block["model_family"] = model_family

        rows.append(
            block.reset_index(drop=True)
        )

    if not rows:
        return None, latest_cp_coefficients

    out = (
        pd.concat(rows, ignore_index=True)
        .set_index(["forecast_date", "factor"])
        .sort_index()
    )

    return out, latest_cp_coefficients


## 47. Horizon-specific unconditional and carry-only benchmarks

These benchmarks use the same horizon target and the same monthly implementation stack.

- `UNCONDITIONAL`: expanding historical mean of the horizon curve-repricing component, using only fully realized targets;
- `CARRY_ONLY`: known horizon funded carry with zero expected curve repricing.


In [ ]:
def build_horizon_unconditional_forecast(
    horizon_months,
    curve_target_h,
    carry_h,
    master_index,
    candidate_dates,
    min_train,
):
    rows = []

    for current_date in candidate_dates:
        cutoff = realization_cutoff_date(
            current_date,
            horizon_months,
            master_index,
        )

        if cutoff is None:
            continue

        hist = curve_target_h.loc[
            curve_target_h.index <= cutoff
        ].dropna()

        if len(hist) < min_train:
            continue

        mu = hist.mean()
        sig = hist.std(ddof=1)
        se = sig / np.sqrt(hist.count())

        block = pd.DataFrame(
            {
                "curve_mean_h": mu,
                "curve_mean_se_h": se,
                "curve_sigma_h": sig,
                "carry_h": carry_h.loc[current_date, FACTOR_NAMES],
            }
        )

        block["total_mean_h"] = (
            block["carry_h"]
            + block["curve_mean_h"]
        )

        block["total_mean"] = block["total_mean_h"] / horizon_months
        block["curve_mean"] = block["curve_mean_h"] / horizon_months
        block["curve_mean_se"] = block["curve_mean_se_h"] / horizon_months
        block["curve_sigma"] = block["curve_sigma_h"] / np.sqrt(horizon_months)
        block["forecast_date"] = current_date
        block["factor"] = block.index

        rows.append(block.reset_index(drop=True))

    return (
        pd.concat(rows, ignore_index=True)
        .set_index(["forecast_date", "factor"])
        .sort_index()
    )


def build_horizon_carry_only_forecast(
    horizon_months,
    carry_h,
    reference_dates,
):
    rows = []

    for current_date in reference_dates:
        if current_date not in carry_h.index:
            continue

        block = pd.DataFrame(
            index=FACTOR_NAMES
        )

        block["carry_h"] = carry_h.loc[
            current_date,
            FACTOR_NAMES,
        ]

        block["curve_mean_h"] = 0.0
        block["curve_mean_se_h"] = 1e-12
        block["curve_sigma_h"] = 1e-12
        block["total_mean_h"] = block["carry_h"]
        block["total_mean"] = block["carry_h"] / horizon_months
        block["curve_mean"] = 0.0
        block["curve_mean_se"] = 1e-12
        block["curve_sigma"] = 1e-12
        block["forecast_date"] = current_date
        block["factor"] = block.index

        rows.append(block.reset_index(drop=True))

    return (
        pd.concat(rows, ignore_index=True)
        .set_index(["forecast_date", "factor"])
        .sort_index()
    )


## 48. Run the full 3M / 6M / 12M empirical-premia grid

The average excess-return target used by the CP first stage is the 2Y–5Y maturity average, consistent with the classical common bond-return-predictor setup.


In [ ]:
v08_forecasts = {}
v08_cp_latest_coefficients = {}
v08_baseline_forecasts = {}
v08_carry_forecasts = {}

for h in PREDICTION_HORIZONS:
    curve_target_h = horizon_factor_targets[h]["curve"]
    carry_h = horizon_factor_targets[h]["carry"]

    avg_asset_excess_h = (
        horizon_asset_data[h]["log_excess"]
        [["ZC2Y", "ZC3Y", "ZC4Y", "ZC5Y"]]
        .mean(axis=1)
    )

    for family in MODEL_FAMILIES:
        forecast, cp_coef = walk_forward_empirical_premia_model(
            horizon_months=h,
            model_family=family,
            curve_target_h=curve_target_h,
            carry_h=carry_h,
            avg_asset_excess_h=avg_asset_excess_h,
            state_panel=state,
            fb_panel=fb_features,
            cp_forward_panel=cp_forward_features,
            master_index=zero_monthly.index,
            min_train=HORIZON_MIN_TRAIN,
            ridge=HORIZON_RIDGE,
        )

        if forecast is not None:
            v08_forecasts[(h, family)] = forecast

        if cp_coef is not None:
            v08_cp_latest_coefficients[(h, family)] = cp_coef

    reference_dates = sorted(
        set().union(
            *[
                set(
                    v08_forecasts[(h, family)]
                    .index
                    .get_level_values("forecast_date")
                )
                for family in MODEL_FAMILIES
                if (h, family) in v08_forecasts
            ]
        )
    )

    reference_dates = pd.DatetimeIndex(reference_dates)

    v08_baseline_forecasts[h] = (
        build_horizon_unconditional_forecast(
            horizon_months=h,
            curve_target_h=curve_target_h,
            carry_h=carry_h,
            master_index=zero_monthly.index,
            candidate_dates=reference_dates,
            min_train=HORIZON_MIN_TRAIN,
        )
    )

    v08_carry_forecasts[h] = (
        build_horizon_carry_only_forecast(
            horizon_months=h,
            carry_h=carry_h,
            reference_dates=reference_dates,
        )
    )

print("Forecast grid built:")
for key, value in v08_forecasts.items():
    print(key, "forecast dates =", value.index.get_level_values(0).nunique())


## 49. OOS predictive diagnostics

For each factor, model and horizon:

- correlation between the predicted and realized horizon curve-repricing component;
- OOS \(R^2\) relative to the expanding unconditional forecast;
- MAE.

The realized targets overlap for 3M/6M/12M under monthly sampling, so these metrics are descriptive. Formal inference on overlapping-horizon regressions should use HAC or block-bootstrap methods.


In [ ]:
def predictive_diagnostics(
    model_forecast,
    uncond_forecast,
    realized_curve_h,
):
    rows = []

    for factor in FACTOR_NAMES:
        model = (
            model_forecast
            .xs(factor, level="factor")
            ["curve_mean_h"]
        )

        uncond = (
            uncond_forecast
            .xs(factor, level="factor")
            ["curve_mean_h"]
        )

        realized = realized_curve_h[factor]

        frame = pd.concat(
            [
                model.rename("model"),
                uncond.rename("uncond"),
                realized.rename("realized"),
            ],
            axis=1,
        ).dropna()

        if len(frame) == 0:
            continue

        sse_model = float(
            ((frame["realized"] - frame["model"]) ** 2).sum()
        )

        sse_uncond = float(
            ((frame["realized"] - frame["uncond"]) ** 2).sum()
        )

        oos_r2 = (
            1.0 - sse_model / sse_uncond
            if sse_uncond > 0
            else np.nan
        )

        rows.append(
            pd.Series(
                {
                    "corr": frame["model"].corr(frame["realized"]),
                    "OOS_R2_vs_uncond": oos_r2,
                    "MAE": (frame["model"] - frame["realized"]).abs().mean(),
                    "n": len(frame),
                },
                name=factor,
            )
        )

    return pd.DataFrame(rows)


predictive_rows = []

for (h, family), forecast in v08_forecasts.items():
    diag = predictive_diagnostics(
        model_forecast=forecast,
        uncond_forecast=v08_baseline_forecasts[h],
        realized_curve_h=horizon_factor_targets[h]["curve"],
    )

    if len(diag) == 0:
        continue

    diag["horizon"] = h
    diag["model"] = family
    diag["factor"] = diag.index

    predictive_rows.append(
        diag.reset_index(drop=True)
    )

v08_predictive_diagnostics = pd.concat(
    predictive_rows,
    ignore_index=True,
)

print("Average OOS predictive diagnostics across factors")
display(
    v08_predictive_diagnostics
    .groupby(["horizon", "model"])
    [["corr", "OOS_R2_vs_uncond", "MAE"]]
    .mean()
    .round(4)
)


## 50. Monthly systematic implementation of each horizon signal

Even though the expected-return model targets an \(H\)-month premium, the portfolio is rebalanced monthly.

The \(H\)-month expected return and its mean-estimation uncertainty are converted to monthly equivalents for risk-scaled sizing. Factor weights are then mapped into maturity weights through the unchanged funded factor-mimicking layer.


In [ ]:
def backtest_forecast_panel(
    forecast_panel,
    cost_bps,
):
    factor_alloc, signal_detail = (
        build_factor_allocations(
            forecast_panel,
            factor_total_returns,
        )
    )

    overlay_w = (
        combine_factor_and_maturity_weights(
            factor_alloc,
            mimic_weights,
            MAX_UNDERLYING_GROSS,
        )
    )

    strategy_ret, overlay_pnl, turnover = (
        apply_overlay_next_month(
            overlay_weights=overlay_w,
            asset_total_returns=zc_returns,
            asset_excess_returns=zc_excess,
            cash_returns=cash_return,
            cost_bps=cost_bps,
            core_returns=None,
            overlay_scale=1.0,
        )
    )

    return {
        "returns": strategy_ret,
        "turnover": turnover,
        "factor_alloc": factor_alloc,
        "overlay_weights": overlay_w,
        "signal_detail": signal_detail,
        "overlay_pnl": overlay_pnl,
    }


v08_backtests = {}

for h in PREDICTION_HORIZONS:
    v08_backtests[(h, "UNCONDITIONAL")] = (
        backtest_forecast_panel(
            v08_baseline_forecasts[h],
            cost_bps=V08_COST_BPS,
        )
    )

    v08_backtests[(h, "CARRY_ONLY")] = (
        backtest_forecast_panel(
            v08_carry_forecasts[h],
            cost_bps=V08_COST_BPS,
        )
    )

    for family in MODEL_FAMILIES:
        if (h, family) not in v08_forecasts:
            continue

        v08_backtests[(h, family)] = (
            backtest_forecast_panel(
                v08_forecasts[(h, family)],
                cost_bps=V08_COST_BPS,
            )
        )


## 51. Strategy comparison across predictors and horizons


In [ ]:
strategy_rows = []

for (h, model), bt in v08_backtests.items():
    r = bt["returns"]
    to = bt["turnover"]

    metrics = performance_metrics(
        r,
        cash=cash_return.reindex(r.index),
        turnover=to,
    )

    metrics["horizon"] = h
    metrics["model"] = model

    strategy_rows.append(metrics)

v08_strategy_results = (
    pd.DataFrame(strategy_rows)
    .set_index(["horizon", "model"])
    .sort_index()
)

print(f"Performance at {V08_COST_BPS:.0f} bps one-way turnover cost")
display(
    v08_strategy_results.round(4)
)

print("Sharpe matrix")
display(
    v08_strategy_results["Sharpe"]
    .unstack("model")
    .round(3)
)


## 52. Conditional value-add relative to horizon-matched unconditional baseline

This isolates whether a predictor changes the **conditional expected-return estimate** in a way that improves the actual monthly implementation.


In [ ]:
value_add_rows = []

for h in PREDICTION_HORIZONS:
    uncond = v08_backtests[(h, "UNCONDITIONAL")]["returns"]

    for family in MODEL_FAMILIES:
        key = (h, family)

        if key not in v08_backtests:
            continue

        model_r = v08_backtests[key]["returns"]

        common = model_r.index.intersection(uncond.index)
        diff = model_r.loc[common] - uncond.loc[common]

        if len(diff) < 12:
            continue

        t_stat = (
            diff.mean()
            / (diff.std(ddof=1) / np.sqrt(len(diff)))
            if diff.std(ddof=1) > 0
            else np.nan
        )

        value_add_rows.append(
            pd.Series(
                {
                    "annualized_mean_spread": diff.mean() * 12.0,
                    "spread_t_stat": t_stat,
                    "n_months": len(diff),
                },
                name=(h, family),
            )
        )

v08_conditional_value_add = pd.DataFrame(value_add_rows)
v08_conditional_value_add.index = pd.MultiIndex.from_tuples(
    v08_conditional_value_add.index,
    names=["horizon", "model"],
)

display(
    v08_conditional_value_add.round(4)
)


## 53. Cost sensitivity across the full empirical-premia grid

No winner is selected here. The table simply checks whether any apparent edge is mechanically dependent on low transaction-cost assumptions.


In [ ]:
cost_rows = []

for cost_bps in COST_SENSITIVITY_BPS:
    for h in PREDICTION_HORIZONS:
        for family in MODEL_FAMILIES:
            key = (h, family)

            if key not in v08_forecasts:
                continue

            bt = backtest_forecast_panel(
                v08_forecasts[key],
                cost_bps=cost_bps,
            )

            r = bt["returns"]

            m = performance_metrics(
                r,
                cash=cash_return.reindex(r.index),
                turnover=bt["turnover"],
            )

            cost_rows.append(
                pd.Series(
                    {
                        "cost_bps": cost_bps,
                        "horizon": h,
                        "model": family,
                        "CAGR": m["CAGR"],
                        "Sharpe": m["Sharpe"],
                        "MaxDD": m["MaxDD"],
                        "Turnover": m["AvgMonthlyTurnover"],
                    }
                )
            )

v08_cost_sensitivity = pd.DataFrame(cost_rows)

print("Sharpe by cost / horizon / model")
display(
    v08_cost_sensitivity
    .pivot_table(
        index=["horizon", "model"],
        columns="cost_bps",
        values="Sharpe",
    )
    .round(3)
)


## 54. Latest Cochrane–Piazzesi-style forward-curve coefficients

The original Cochrane–Piazzesi result is associated with a tent-shaped combination of forward rates. The coefficients below are **walk-forward estimates from the current sample**, not hard-coded academic coefficients.

For 3M and 6M they are horizon extensions; 12M is the literature-aligned horizon.


In [ ]:
cp_coef_rows = []

for (h, family), coef in v08_cp_latest_coefficients.items():
    if family != "CP":
        continue

    row = coef.copy()
    row["horizon"] = h
    cp_coef_rows.append(row)

if cp_coef_rows:
    v08_cp_latest_table = (
        pd.DataFrame(cp_coef_rows)
        .set_index("horizon")
        .sort_index()
    )

    display(
        v08_cp_latest_table.round(4)
    )

    forward_cols = [
        c for c in v08_cp_latest_table.columns
        if c.startswith("f")
    ]

    ax = (
        v08_cp_latest_table[forward_cols]
        .T
        .plot(
            marker="o",
            figsize=(10, 5),
        )
    )

    ax.set_title(
        "Latest CP-style forward-rate coefficients by forecast horizon"
    )
    ax.set_xlabel("Forward-rate input")
    ax.set_ylabel("Coefficient")

    plt.show()


## 55. Latest allocations by model and horizon

This is useful for understanding whether different empirical asset-pricing predictors are producing economically distinct Level/Slope/Curvature views rather than merely small numerical variations of the same allocation.


In [ ]:
latest_alloc_rows = []

for (h, model), bt in v08_backtests.items():
    if model in {"UNCONDITIONAL", "CARRY_ONLY"}:
        continue

    alloc = bt["factor_alloc"]

    if len(alloc) == 0:
        continue

    latest = alloc.iloc[-1].copy()
    latest["horizon"] = h
    latest["model"] = model
    latest["forecast_date"] = alloc.index[-1]

    latest_alloc_rows.append(latest)

v08_latest_allocations = (
    pd.DataFrame(latest_alloc_rows)
    .set_index(["horizon", "model"])
    .sort_index()
)

display(
    v08_latest_allocations.round(3)
)


## 56. Research interpretation rules for v0.8

The v0.8 grid is designed to answer distinct empirical questions.

### A. Fama–Bliss

If `FB` or `FB_STATE` improves OOS return forecasts and strategy value relative to `UNCONDITIONAL`, the forward spread contains useful conditional bond-premium information.

### B. Cochrane–Piazzesi style

If `CP` improves across several factor portfolios, that supports the idea of a common return-forecasting component that is distinct from the Level/Slope/Curvature **risk-factor decomposition**.

### C. Forecast horizon

If 6M or 12M materially dominates 3M, the data are telling us that expected rates premia evolve more slowly than monthly return noise. Monthly rebalancing can still be retained for implementation.

### D. Generic state versus asset-pricing predictors

`BASE_STATE` is the v0.7 benchmark. If `FB` / `CP` outperform it, the correct upgrade is economic signal design, not threshold tuning.

### E. Combined models

`FB_STATE` and `CP_STATE` test whether macro/curve state adds information beyond the term structure itself. They should not automatically be preferred merely because they contain more predictors.

### F. No automatic winner selection

A model should be considered promising only if the result is coherent across:

- predictive OOS diagnostics;
- economic performance;
- cost sensitivity;
- subperiods;
- neighboring horizons;
- economically interpretable exposures.


## 57. References and scope

The signal layer is motivated by:

- Fama, Eugene F. and Robert R. Bliss (1987), *The Information in Long-Maturity Forward Rates*.
- Cochrane, John H. and Monika Piazzesi (2005), *Bond Risk Premia*, American Economic Review 95(1), 138–160.

The current zero-coupon series are the Federal Reserve / FRED fitted Treasury curve already used throughout this project.

The 3M and 6M CP implementations are intentional horizon extensions. The 12M implementation is the closest to the canonical annual-return setup.


## 58. Save v0.8 research outputs


In [ ]:
OUTPUT_DIR_V08 = Path(
    "./rates_factor_timing_outputs_v08"
)

OUTPUT_DIR_V08.mkdir(
    exist_ok=True
)

fb_12m_diagnostics.to_csv(
    OUTPUT_DIR_V08 / "fama_bliss_12m_diagnostics.csv"
)

v08_predictive_diagnostics.to_csv(
    OUTPUT_DIR_V08 / "predictive_diagnostics.csv",
    index=False,
)

v08_strategy_results.to_csv(
    OUTPUT_DIR_V08 / "strategy_results.csv"
)

v08_conditional_value_add.to_csv(
    OUTPUT_DIR_V08 / "conditional_value_add.csv"
)

v08_cost_sensitivity.to_csv(
    OUTPUT_DIR_V08 / "cost_sensitivity.csv",
    index=False,
)

v08_latest_allocations.to_csv(
    OUTPUT_DIR_V08 / "latest_factor_allocations.csv"
)

if "v08_cp_latest_table" in globals():
    v08_cp_latest_table.to_csv(
        OUTPUT_DIR_V08 / "latest_cp_coefficients.csv"
    )

for (h, family), forecast in v08_forecasts.items():
    forecast.to_csv(
        OUTPUT_DIR_V08
        / f"forecast_{h}m_{family.lower()}.csv"
    )

print(
    "Saved v0.8 outputs to:",
    OUTPUT_DIR_V08.resolve(),
)
